# GRPO Fine-Tuning on the AWS RL Environment

**Curriculum-driven GRPO**, built on top of the existing SFT LoRA adapter ([`Sizzing/aws-rl-sft-qwen25coder3b-adapter`](https://huggingface.co/Sizzing/aws-rl-sft-qwen25coder3b-adapter)). The priority-queue curriculum in [`server/services/curriculum.py`](https://github.com/UdayKiranPadhy/aws-rl-env/blob/master/server/services/curriculum.py) picks each training task based on novelty, weakness, and spaced repetition; TRL's `GRPOTrainer` owns the generate/score/update loop; rewards flow back into the curriculum through the reward function to close the mastery loop.

Hyperparameter-tuned with Optuna, logged to Weights & Biases, checkpointed for safe resume, and published to a **separate** HuggingFace Hub repo so both the SFT and GRPO adapters coexist side-by-side.

### Architecture
```
Hosted env server (Docker, AWS_RL_ENV_POOL_SIZE=8)
        ▲ HTTPS tunnel
        │                                Colab / Kaggle VM (T4)
        │                                 └─ main python
        │  8-way httpx pool (rewards)         ├─ Unsloth: base + SFT adapter (trainable)
        ├────────────────────────       │
        │                                      ├─ TRL GRPOTrainer
        │  8-way GrpoPool WS (eval)            │    ├─ train_ds = stream from Curriculum
        └────────────────────────       │    ├─ G generations / prompt
                                               │    ├─ reward_fn:
                                               │    │    a) score via /reset + /step
                                               │    │    b) curriculum.record_result(...)
                                               │    └─ group-advantage + PPO-clip on LoRA
                                               ├─ Optuna (sqlite, resumable)
                                               └─ push → Sizzing/aws-rl-grpo-qwen25coder3b-adapter
```

### Why curriculum-driven GRPO?
Plain dataset iteration is agnostic to the model’s current weaknesses. The curriculum surfaces novel + weak tasks more often, spaced-repeats mastered ones, and promotes tier only when the agent demonstrates sustained competence. GRPO's group-relative advantage + the curriculum's per-task mastery signal compose: a group of G completions on a hard task yields a high-variance advantage signal exactly when it matters.

### Headline metrics (reported in the eval cells)
| Axis                 | SFT baseline | GRPO target | Stretch |
|----------------------|-------------:|------------:|--------:|
| `env_reward_mean`    |        ~0.85 |        ≥0.92 |   ≥0.97 |
| `env_success_rate`   |         ~85% |         ≥92% |    ≥97% |
| `recovery_rate`      |         ~40% |         ≥60% |    ≥75% |
| `drift_repair_rate`  |         ~30% |         ≥55% |    ≥70% |
| `hints_per_solved`   |         ~0.8 |        ≤0.5 |    ≤0.2 |
| `steps_to_solve`     |           ~6 |          ≤5 |      ≤4 |

## 1 · Install dependencies

Mirrors the SFT notebook's pinning strategy. Key constraints: `trl>=0.16` for `GRPOTrainer`, `transformers>=4.50,<5.0` for Unsloth compatibility (the SFT notebook ran into this too), `--force-reinstall --no-deps` on `transformers` so TRL doesn't pull an incompatible version.

In [1]:
%pip install -q --upgrade pip
%pip install -q "unsloth"
%pip install -q "trl>=0.18.2,<=0.24.0,!=0.19.0" "peft" "accelerate" "datasets" "bitsandbytes"
%pip install -q "transformers>=4.50,<5.0" --force-reinstall --no-deps
%pip install -q "optuna"
%pip install -q "httpx" "websockets" "pyyaml" "python-dotenv"
%pip install -q "huggingface_hub>=0.34,<1.0"
%pip install -q "openenv-core"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.4 MB/s eta 0:00:00


### 1b · Logging & warning suppression

Set up structured logging and suppress known benign warnings (PEFT config keys, notebook escape sequences, transformers do_sample advisory).

In [3]:
import logging
import sys
import warnings

# ── Structured logging ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger("grpo")

# ── Suppress known benign warnings ──────────────────────────────────
# PEFT adapter was saved with a newer version; extra keys are safely ignored.
warnings.filterwarnings(
    "ignore",
    message=r"Unexpected keyword arguments.*for class LoraConfig",
    category=UserWarning,
)
# The notebook package itself has an invalid escape sequence in its banner.
warnings.filterwarnings(
    "ignore",
    message=r"invalid escape sequence",
    category=SyntaxWarning,
)
# Transformers logs "do_sample is False but temperature is set" when we
# intentionally use greedy decoding with temperature=1.0 (the default).
warnings.filterwarnings(
    "ignore",
    message=r".*`do_sample` is set to `False`.*`temperature`.*",
    category=UserWarning,
)

log.info("Logging configured; benign warnings suppressed.")


14:52:11 | INFO    | grpo | Logging configured; benign warnings suppressed.


## 2 · Clone the AWS RL env repo

We need the `client.AwsRlEnv` class, `models.py` pydantic types, `scripts/grpo_pool.py`, the curriculum loader, and the task YAMLs — these power the multi-step evaluator. We do **not** install or run any env server locally; the env is hosted elsewhere.

In [4]:
from __future__ import annotations
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/UdayKiranPadhy/aws-rl-env.git"

# Detect host in priority order: Kaggle first (it often has /content too),
# then Colab, then fall back to CWD for local runs.
if Path("/kaggle/working").exists():
    REPO_DIR = Path("/kaggle/working/aws-rl-env")
elif Path("/content").exists():
    REPO_DIR = Path("/content/aws-rl-env")
else:
    REPO_DIR = (Path.cwd() / "aws-rl-env").resolve()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# If a prior partial clone left no models.py, redo it.
if not (REPO_DIR / "models.py").exists():
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

assert (REPO_DIR / "models.py").exists(), f"models.py missing under {REPO_DIR}"
log.info("Repo at %s — ready. sys.path[0]=%s", REPO_DIR, sys.path[0])


14:52:16 | INFO    | grpo | Repo at /content/aws-rl-env — ready. sys.path[0]=/content/aws-rl-env


## 3 · Runtime detection

Matches the SFT notebook so we pick `fp16` on T4 (bf16 unsupported) and `bf16` on A100/H100. Also sets the PyTorch allocator env var that cut OOMs on long Unsloth runs.

In [5]:
from dataclasses import dataclass
import torch

IS_COLAB  = True
IS_KAGGLE = False

OUT_DIR = Path("/content/out") if IS_COLAB else Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")


@dataclass(frozen=True)
class Runtime:
    gpu_name: str
    use_fp16: bool
    use_bf16: bool


def detect_runtime() -> Runtime:
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. This notebook needs CUDA (T4/A100/H100).")
    name = torch.cuda.get_device_name(0)
    is_t4 = "T4" in name
    return Runtime(gpu_name=name, use_fp16=is_t4, use_bf16=not is_t4)


RT = detect_runtime()
log.info("GPU: %s | fp16=%s bf16=%s | OUT_DIR=%s", RT.gpu_name, RT.use_fp16, RT.use_bf16, OUT_DIR)

14:52:29 | INFO    | grpo | GPU: Tesla T4 | fp16=True bf16=False | OUT_DIR=/content/out


## 4 · Training configuration

One frozen `TrainingConfig` dataclass holds every tunable hyperparameter. Optuna hands trials a mutated copy of this same dataclass, so the trial path and the final-run path go through identical code.

Separate `ModelSpec` and `PathsSpec` keep non-tunable identifiers out of the search space.

In [6]:
from dataclasses import replace


@dataclass(frozen=True)
class ModelSpec:
    base_model:   str = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
    sft_adapter:  str = "Sizzing/aws-rl-sft-qwen25coder3b-adapter"
    grpo_adapter: str = "Sizzing/aws-rl-grpo-qwen25coder3b-adapter"   # NEW repo — SFT repo untouched
    dataset_repo: str = "Sizzing/aws-rl-sft"
    max_seq_length: int = 3072


@dataclass(frozen=True)
class TrainingConfig:
    # GRPO knobs (Optuna-tunable)
    learning_rate:   float = 5e-6 # AdamW learning rate;
    beta:            float = 0.04    # KL coef vs. reference (frozen SFT adapter)
    num_generations: int   = 8       # G in GRPO
    temperature:     float = 0.9 # Sampling temp during generation
    top_p:           float = 0.95 # Nucleus sampling cutoff
    max_prompt_length:     int = 512 # Hard truncate on the prompt side
    max_completion_length: int = 768 # Max tokens on the completion side;
    per_device_train_batch_size: int = 2 # Batch per GPU
    gradient_accumulation_steps: int = 8 # 	Effective batch = pdtbs × grad_accum (= 16 now)
    num_train_epochs:      int = 1     # ignored for IterableDataset; max_steps drives termination
    save_steps:            int = 25 # How often the final run writes a checkpoint-N
    save_total_limit:      int = 3 # Max number of checkpoints to keep; older ones get deleted
    eval_steps:            int = 50 # How often to run evaluation
    warmup_ratio:          float = 0.05
    seed:                  int = 42 # Seeds GRPOConfig + Optuna's TPESampler


@dataclass(frozen=True)
class PipelineConfig:
    env_pool_size: int = 8
    n_trials:      int = 4  # Optuna trials for hyperparameter search; each trial trains a GRPO agent from the same SFT starting point, but with different hyperparameters.
    trial_max_steps:  int = 10 # max_steps for each Optuna trial; keeps trial runtime manageable and encourages faster feedback on hyperparameter choices. The final evaluation after all trials will use a longer max_steps to allow the best trial more time to shine.
    final_max_steps:  int = 35  # GRPO steps for the post-Optuna final run
    val_subset_size:  int = 20
    eval_reserve_cap: int = 100
    # 12-task pool used for both Optuna trial training and trial scoring.
    # Tuple-of-tuples keeps the dataclass frozen/hashable.
    optuna_tier_counts: tuple = (
        ("warmup", 2), ("beginner", 2), ("intermediate", 2),
        ("advanced", 3), ("expert", 3),
    )


MODEL = ModelSpec()
TRAIN = TrainingConfig()
PIPE  = PipelineConfig()

log.info("ModelSpec: %s", MODEL)
log.info("TrainingConfig defaults: %s", TRAIN)
log.info("PipelineConfig: %s", PIPE)


14:52:32 | INFO    | grpo | ModelSpec: ModelSpec(base_model='unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit', sft_adapter='Sizzing/aws-rl-sft-qwen25coder3b-adapter', grpo_adapter='Sizzing/aws-rl-grpo-qwen25coder3b-adapter', dataset_repo='Sizzing/aws-rl-sft', max_seq_length=3072)
14:52:32 | INFO    | grpo | TrainingConfig defaults: TrainingConfig(learning_rate=5e-06, beta=0.04, num_generations=8, temperature=0.9, top_p=0.95, max_prompt_length=512, max_completion_length=768, per_device_train_batch_size=2, gradient_accumulation_steps=8, num_train_epochs=1, save_steps=25, save_total_limit=3, eval_steps=50, warmup_ratio=0.05, seed=42)
14:52:32 | INFO    | grpo | PipelineConfig: PipelineConfig(env_pool_size=8, n_trials=4, trial_max_steps=10, final_max_steps=35, val_subset_size=20, eval_reserve_cap=100, optuna_tier_counts=(('warmup', 2), ('beginner', 2), ('intermediate', 2), ('advanced', 3), ('expert', 3)))


## 5 · Authenticate: HF Hub + env URL

In [7]:
def _load_secret(name: str) -> str:
    """Read a secret from Colab userdata, Kaggle UserSecrets, or the env."""
    if IS_COLAB:
        from google.colab import userdata
        try: return userdata.get(name)
        except Exception: pass
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        try: return UserSecretsClient().get_secret(name)
        except Exception: pass
    val = os.environ.get(name)
    if not val:
        raise RuntimeError(f"Secret {name!r} is missing. Set it in Colab/Kaggle secrets or as an env var.")
    return val


HF_TOKEN     = _load_secret("HF_TOKEN")
ENV_BASE_URL = _load_secret("AWS_RL_ENV_BASE_URL").rstrip("/")

from huggingface_hub import login as hf_login, HfApi

hf_login(token=HF_TOKEN, add_to_git_credential=False)


def verify_hub_write_scope(repo_id: str) -> None:
    """Ensure the HF token can create repos under the target org before training.

    Without this, we'd discover permission failures *after* a multi-hour run.
    """
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=repo_id, private=True, exist_ok=True, repo_type="model")
    probe = OUT_DIR / ".hub_write_probe.txt"
    probe.write_text("ok")
    api.upload_file(path_or_fileobj=str(probe), path_in_repo=".hub_write_probe.txt",
                    repo_id=repo_id, commit_message="probe: verify write scope")
    probe.unlink()


verify_hub_write_scope(MODEL.grpo_adapter)
log.info("All secrets loaded; HF write access to %s verified.", MODEL.grpo_adapter)


No files have been modified since last commit. Skipping to prevent empty commit.


14:52:38 | WARNING | huggingface_hub.hf_api | No files have been modified since last commit. Skipping to prevent empty commit.
14:52:38 | INFO    | grpo | All secrets loaded; HF write access to Sizzing/aws-rl-grpo-qwen25coder3b-adapter verified.


## 6 · Connect to the hosted env + health check

The env server runs **outside** this VM. We only assert reachability and that its internal pool has 8 slots (required for parallel rollouts). If either fails, we stop **before** loading the model.

In [8]:
import httpx


def probe_env_http(base_url: str) -> dict:
    """Cheap reachability check: GET /schema. Raises on HTTP error.

    Does NOT try to validate pool size from /state — AwsRlState doesn't
    expose it. Pool capacity is verified later via a real GrpoPool
    connection attempt (§6b) which is the only honest way to check.
    """
    with httpx.Client(base_url=base_url, timeout=10.0) as c:
        schema = c.get("/schema").raise_for_status().json()
    return {"schema": schema}


probe = probe_env_http(ENV_BASE_URL)
log.info("Env reachable. Action schema keys: %s", list(probe['schema'].keys()))
log.info("POOL_SIZE=%d assumed — verified by GrpoPool connection below.", PIPE.env_pool_size)

14:52:44 | INFO    | httpx | HTTP Request: GET https://sizzing-aws-rl-env.hf.space/schema "HTTP/1.1 200 OK"
14:52:44 | INFO    | grpo | Env reachable. Action schema keys: ['action', 'observation', 'state']
14:52:44 | INFO    | grpo | POOL_SIZE=8 assumed — verified by GrpoPool connection below.


## 7 · Curriculum-driven prompt stream + fixed val / reserve sets

Instead of feeding a static dataset, we stream prompts from the repo's [`Curriculum`](https://github.com/UdayKiranPadhy/aws-rl-env/blob/master/server/services/curriculum.py) — the same priority-queue curriculum the env already implements:

- **novelty bonus** for untried tasks (+100)
- **weakness weighting** `(1 − recent_success_rate) × 50` per task
- **spaced repetition** on graduated tasks at 3→66→12→24→48 episode intervals
- **recency penalty** to avoid drilling the same task back-to-back
- **tier promotion** with fast-track when success rate crosses threshold

TRL's `GRPOTrainer` accepts a `datasets.IterableDataset`; we wrap `curriculum.next_task()` in a generator and feed it in. The feedback loop closes inside the reward function (next cell): after scoring a group of G completions for a task, it calls `curriculum.record_result(task, achieved, mean_reward)`, which updates mastery, promotes tiers, and re-ranks the priority queue for the next step.

`VAL_DS` (fixed 20-row subset from the HF dataset's `validation` split) and `RESERVE_DS` (up to 100 rows from `reserve`) stay fixed for comparable before/after evaluation.

In [9]:
from datasets import load_dataset, Dataset
from models import Task, TaskDifficulty
from server.services.curriculum import Curriculum, load_tier
import yaml


def build_task_map(tasks_dir: Path) -> dict[int, Task]:
    """Flatten every task YAML into a dict keyed by task_id.

    The reward function only has task_id to work with; this map lets it
    recover the full Task object needed to serialise over HTTP to /reset.
    """
    m: dict[int, Task] = {}
    for tier in TaskDifficulty:
        try:
            for t in load_tier(tier, tasks_dir):
                m[int(t.task_id)] = t
        except FileNotFoundError:
            continue
    return m


def load_drift_task_ids(tasks_dir: Path) -> set[int]:
    """Drift tasks live in drift.yaml and get folded into the EXPERT tier by
    the curriculum loader. We scan the file directly so we can still identify
    them for drift_repair_rate in multi-step eval.
    """
    ids: set[int] = set()
    drift_path = tasks_dir / "drift.yaml"
    if drift_path.exists():
        with open(drift_path) as f:
            for entry in (yaml.safe_load(f) or []):
                ids.add(int(entry["task_id"]))
    return ids


SYSTEM_PROMPT = (
    "You are an expert AWS Operations agent. You operate a simulated AWS cloud by "
    "emitting one AWS CLI command per turn. The conversation may include prior "
    "commands and their outputs from earlier in this episode — use them to decide "
    "your next move.\n\n"
    "First reason about your next move inside a <think>...</think> block: identify "
    "the AWS service, the right subcommand, required arguments, and any constraints "
    "from the task. Keep the reasoning concise (a few short sentences).\n\n"
    "After </think>, on a NEW LINE, output EXACTLY ONE AWS CLI command starting "
    "with \"aws \". The command line must contain only the command — no markdown, "
    "no backticks, no quotes around it, and no trailing commentary."
)


def task_to_row(task: Task) -> dict:
    """Convert a Task into the (prompt, task_id, difficulty) schema the
    curriculum-aware sampler reads. `remove_unused_columns=False` in the
    trainer config keeps `difficulty` on the dataset; reward funcs accept
    **kw so the extra column is harmlessly ignored there.
    """
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"TASK: {task.description}"},
        ],
        "task_id":    int(task.task_id),
        "difficulty": task.difficulty.value,
    }


def make_curriculum_dataset(curriculum: Curriculum, n_rows: int) -> Dataset:
    """Pre-materialise N curriculum-picked prompts as a finite `datasets.Dataset`.

    TRL's GRPOTrainer explicitly rejects IterableDataset (see TRL issue #3213:
    ``NotImplementedError: Iterable datasets are not yet supported``). That
    kills the original on-demand `curriculum.next_task()` streaming design —
    the trainer must see a Dataset with a known `__len__`.

    Compromise: sample `n_rows` prompts up front. The curriculum's novelty +
    weakness + recency heuristics still apply *across the draw* (every
    `next_task()` pops the current top-of-queue and ages neighbouring tasks
    via recency), so we get a sensibly ordered warm-start sample — not
    uniform-random. What we lose is live re-ranking between steps: mastery
    updates made by the reward function during training don't feed back
    into selection until a fresh dataset is built. `curriculum.record_result`
    still runs inside the reward function so mastery metrics remain accurate
    for end-of-run stats.

    Size `n_rows` to `max_steps * per_device_train_batch_size *
    gradient_accumulation_steps`, plus a buffer so `num_train_epochs=1`
    never exhausts the dataset before `max_steps` terminates training.
    """
    return Dataset.from_list(
        [task_to_row(curriculum.next_task()) for _ in range(n_rows)]
    )


def make_full_curriculum_dataset(tasks_dir: Path) -> Dataset:
    """Build a *superset* dataset spanning every difficulty tier.

    This replaces `make_curriculum_dataset` for curriculum-driven training:
    instead of drawing N prompts up front (which freezes the training data
    to whatever tier the curriculum happened to start in), we materialise
    EVERY task with its `difficulty` tag and let `CurriculumTierSampler`
    dynamically pick indices from the curriculum's current tier at
    iteration time. Tier promotion inside the reward callback then
    immediately affects which prompts the next batch pulls.
    """
    rows: list[dict] = []
    for tier in TaskDifficulty:
        for task in load_tier(tier, tasks_dir):
            rows.append(task_to_row(task))
    if not rows:
        raise RuntimeError(f"No tasks found under {tasks_dir}")
    ds = Dataset.from_list(rows)
    log.info(
        "Full curriculum dataset: %d rows across tiers %s",
        len(ds),
        sorted({r["difficulty"] for r in rows}),
    )
    return ds


def make_optuna_task_dataset(task_map: dict[int, Task],
                             tier_counts: tuple,
                             seed: int = 42) -> Dataset:
    """Deterministic small task pool for Optuna trials.

    Picks N tasks per tier from `task_map` (sorted by task_id then shuffled
    with a fixed seed so the pool is identical across re-runs). Returns a
    Dataset with the same row schema as `make_full_curriculum_dataset`.
    Used both as trial train_ds (curriculum sampler cycles within tier
    pools) and as the single-step eval set for `trial_objective`.
    """
    import random
    rng = random.Random(seed)
    rows: list[dict] = []
    for tier_name, n in tier_counts:
        pool = sorted(
            (t for t in task_map.values() if t.difficulty.value == tier_name),
            key=lambda t: int(t.task_id),
        )
        rng.shuffle(pool)
        for t in pool[:n]:
            rows.append(task_to_row(t))
    if not rows:
        raise RuntimeError("Optuna pool came out empty — check tier_counts")
    ds = Dataset.from_list(rows)
    log.info("Optuna task pool: %d rows by tier=%s",
             len(ds), {k: n for k, n in tier_counts})
    return ds


def _load_raw_dataset(dataset_repo: str) -> dict:
    """Load the HF dataset once; cached for reuse by val + reserve builders."""
    log.info("Loading dataset from %s …", dataset_repo)
    return load_dataset(dataset_repo, token=HF_TOKEN)


_RAW_DS_CACHE: dict = {}


def _get_raw(dataset_repo: str):
    if dataset_repo not in _RAW_DS_CACHE:
        _RAW_DS_CACHE[dataset_repo] = _load_raw_dataset(dataset_repo)
    return _RAW_DS_CACHE[dataset_repo]


def build_val_dataset(dataset_repo: str, task_map: dict[int, Task],
                      val_size: int, seed: int = 42) -> Dataset:
    """Fixed validation subset for comparable before/after eval."""
    raw = _get_raw(dataset_repo)
    val_single = raw["validation"].filter(
        lambda r: r["step_idx"] == 0 and int(r["task_id"]) in task_map
    )
    val = val_single.shuffle(seed=seed).select(
        range(min(val_size, len(val_single)))
    )
    return val.map(
        lambda r: {"prompt": r["messages"][:2], "task_id": int(r["task_id"])},
        remove_columns=[c for c in val.column_names if c not in ("prompt", "task_id")],
    )


def build_reserve_dataset(dataset_repo: str,
                          task_map: dict[int, Task]) -> Dataset | None:
    """Reserve split for the multi-step eval."""
    raw = _get_raw(dataset_repo)
    if "reserve" not in raw:
        return None
    reserve_single = raw["reserve"].filter(
        lambda r: r["step_idx"] == 0 and int(r["task_id"]) in task_map
    )
    return reserve_single.map(
        lambda r: {"prompt": r["messages"][:2], "task_id": int(r["task_id"])},
        remove_columns=[c for c in reserve_single.column_names
                         if c not in ("prompt", "task_id")],
    )


_tasks_dir = REPO_DIR / "server" / "services" / "tasks"
TASK_MAP = build_task_map(_tasks_dir)
DRIFT_TASK_IDS = load_drift_task_ids(_tasks_dir)
VAL_DS = build_val_dataset(MODEL.dataset_repo, TASK_MAP,
                            PIPE.val_subset_size, TRAIN.seed)
RESERVE_DS = build_reserve_dataset(MODEL.dataset_repo, TASK_MAP)
OPTUNA_DS = make_optuna_task_dataset(TASK_MAP, PIPE.optuna_tier_counts, TRAIN.seed)

log.info("TASK_MAP: %d tasks across %d tiers", len(TASK_MAP), len({t.difficulty for t in TASK_MAP.values()}))
log.info("DRIFT_TASK_IDS: %d drift tasks", len(DRIFT_TASK_IDS))
log.info("VAL_DS: %d rows (fixed, for before/after comparison)", len(VAL_DS))
log.info("RESERVE_DS: %d rows (multi-step eval)", len(RESERVE_DS) if RESERVE_DS else 0)
log.info("OPTUNA_DS: %d rows (used for trial training + trial val eval)", len(OPTUNA_DS))


14:52:47 | INFO    | numexpr.utils | NumExpr defaulting to 2 threads.
14:52:48 | INFO    | datasets | TensorFlow version 2.19.0 available.
14:52:48 | INFO    | datasets | JAX version 0.7.2 available.
14:53:02 | INFO    | server.services.curriculum | Loaded 25 warmup tasks total
14:53:02 | INFO    | server.services.curriculum | Loaded 25 beginner tasks total
14:53:02 | INFO    | server.services.curriculum | Loaded 25 intermediate tasks total
14:53:02 | INFO    | server.services.curriculum | Loaded 25 advanced tasks total
14:53:03 | INFO    | server.services.curriculum | Loaded 9 supplementary expert tasks from drift.yaml
14:53:03 | INFO    | server.services.curriculum | Loaded 33 expert tasks total
14:53:03 | INFO    | grpo | Loading dataset from Sizzing/aws-rl-sft …


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.92M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/194k [00:00<?, ?B/s]

data/reserve-00000-of-00001.parquet:   0%|          | 0.00/261k [00:00<?, ?B/s]

...

Filter:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

14:53:13 | INFO    | grpo | Optuna task pool: 12 rows by tier={'warmup': 2, 'beginner': 2, 'intermediate': 2, 'advanced': 3, 'expert': 3}
14:53:13 | INFO    | grpo | TASK_MAP: 133 tasks across 5 tiers
14:53:13 | INFO    | grpo | DRIFT_TASK_IDS: 9 drift tasks
14:53:13 | INFO    | grpo | VAL_DS: 20 rows (fixed, for before/after comparison)
14:53:13 | INFO    | grpo | RESERVE_DS: 112 rows (multi-step eval)
14:53:13 | INFO    | grpo | OPTUNA_DS: 12 rows (used for trial training + trial val eval)


## 8 · Reward functions + curriculum feedback

Three reward functions passed to `GRPOTrainer.reward_funcs`:

| Reward           | Weight | Signal                                                     |
|------------------|-------:|------------------------------------------------------------|
| `env_reward`     |   1.0  | Real env reward from `/reset` + `/step` against the task   |
| `format_reward`  |  0.15  | 1.0 if completion starts with `aws `, else 0.0             |
| `length_reward`  |  0.05  | Soft length prior: 1.0 ≤120 chars, decays to 0.0 by 400    |

`env_reward` also closes the curriculum loop. TRL emits the batch as `batch_size × num_generations` flattened completions with `task_id` repeated G times per prompt. We group by task_id, and for each group call `curriculum.record_result(task, achieved=any(r≥1.0), reward=mean)`. That one line is the bridge: TRL owns iteration, the curriculum owns task selection, and the reward function owns the feedback edge. No custom training loop; all the quality-of-life of TRL (checkpoint, resume, Optuna) is preserved.

Thread safety: env HTTP calls run on the pool threads, but reward aggregation + `record_result` both happen on the main thread after the pool join, so no locking is needed.


In [10]:
import asyncio
import re
import threading
import traceback
from collections import defaultdict
from typing import Callable, Optional

from client import AwsRlEnv
from models import AwsRlAction
from websockets.exceptions import ConnectionClosed


_THINK_BLOCK = re.compile(r"<think\b[^>]*>.*?</think>", re.DOTALL | re.IGNORECASE)
_OPEN_THINK  = re.compile(r"<think\b[^>]*>.*", re.DOTALL | re.IGNORECASE)


def extract_aws_command(raw: str) -> str:
    # Drop any balanced <think>...</think> spans first.
    cleaned = _THINK_BLOCK.sub("", raw)
    # If a <think> is still open (no matching </think>), everything after
    # it is reasoning-in-progress, not a command — cut it.
    cleaned = _OPEN_THINK.sub("", cleaned)
    for line in cleaned.splitlines():
        line = line.strip().strip("`").strip()
        if line.startswith("aws "):
            return line
    return "aws help"


class EnvRewardClient:
    """Persistent-WebSocket reward client against a pooled env server.

    Why WebSocket and not HTTP /reset + /step: under `AWS_RL_ENV_POOL_SIZE>1`
    the server's HTTP path uses an env *factory* — every request builds a
    fresh `AwsRlEnvironment` from the pool factory, so `/step` on request 2
    lands on a different env than `/reset` on request 1 and trips
    ``assert self._episode is not None, "Call reset() before step()"``.
    Only WebSocket sessions hold a MiniStack slot across calls.

    Design:
      - Dedicated background thread owns an asyncio loop.
      - On startup, N AwsRlEnv WebSocket sessions connect in parallel and
        sit in an asyncio.Queue acting as a free-list.
      - score_batch() is synchronous (TRL calls reward funcs sync): it
        submits one async task per (task, command) pair to the loop via
        run_coroutine_threadsafe, each task acquires a free session from
        the queue, does reset+step, returns the env to the queue.
      - Reconnect-on-failure: Cloudflare tunnels can idle-close WS
        sessions, but the client keeps a reference to the (now-closed)
        socket so OpenEnv's _ensure_connected() is a no-op. We catch
        ConnectionClosed explicitly, call disconnect(), reconnect, and
        retry once before giving up.

    Counters (success / timeout / conn_err / reconnect) let the trainer
    log env health without inspecting internal state. `last_error`
    captures the most recent exception. `verbose_errors=True` prints
    full tracebacks per failure (noisy — only enable while debugging).
    """

    def __init__(
        self,
        base_url: str,
        pool_size: int = 8,
        timeout_s: float = 60.0,
        verbose_errors: bool = False,
    ):
        self.base_url = base_url
        self.pool_size = pool_size
        self.timeout_s = timeout_s
        self.verbose_errors = verbose_errors
        self.success = 0
        self.timeout = 0
        self.conn_err = 0
        self.reconnect = 0
        self.last_error: Optional[str] = None
        self._loop: Optional[asyncio.AbstractEventLoop] = None
        self._thread: Optional[threading.Thread] = None
        self._queue: Optional[asyncio.Queue] = None
        self._envs: list = []
        self._ready = threading.Event()
        self._setup_error: Optional[BaseException] = None
        self._start()

    def _start(self) -> None:
        def run():
            loop = asyncio.new_event_loop()
            self._loop = loop
            asyncio.set_event_loop(loop)
            try:
                loop.run_until_complete(self._setup())
            except BaseException as e:
                self._setup_error = e
                self._ready.set()
                return
            self._ready.set()
            loop.run_forever()

        self._thread = threading.Thread(target=run, daemon=True, name="env-reward")
        self._thread.start()
        self._ready.wait()
        if self._setup_error is not None:
            raise RuntimeError(
                f"EnvRewardClient failed to connect {self.pool_size} WS sessions "
                f"to {self.base_url}: {self._setup_error!r}"
            )

    async def _setup(self) -> None:
        self._queue = asyncio.Queue(self.pool_size)
        self._envs = [AwsRlEnv(base_url=self.base_url) for _ in range(self.pool_size)]
        try:
            await asyncio.gather(*(e.connect() for e in self._envs))
        except BaseException:
            await asyncio.gather(
                *(e.close() for e in self._envs), return_exceptions=True
            )
            raise
        for e in self._envs:
            self._queue.put_nowait(e)

    async def _reconnect(self, env) -> None:
        """Discard the dead socket and open a fresh WS session in-place."""
        self.reconnect += 1
        try:
            await env.disconnect()
        except Exception:
            pass
        await env.connect()

    async def _reset_and_step(self, env, task: Task, command: str) -> float:
        await asyncio.wait_for(env.reset(task=task), timeout=self.timeout_s)
        res = await asyncio.wait_for(
            env.step(AwsRlAction(command=command)), timeout=self.timeout_s
        )
        return float(res.reward)

    async def _score_one(self, task: Task, command: str) -> float:
        env = await self._queue.get()
        try:
            try:
                reward = await self._reset_and_step(env, task, command)
                self.success += 1
                return reward
            except ConnectionClosed as e:
                # Cloudflare / server idle-closed the socket. Reconnect and retry.
                self.last_error = f"reconnect after {type(e).__name__}: {e}"
                if self.verbose_errors:
                    print(f"[reward] {self.last_error} — reconnecting")
                await self._reconnect(env)
                reward = await self._reset_and_step(env, task, command)
                self.success += 1
                return reward
        except asyncio.TimeoutError:
            self.timeout += 1
            self.last_error = "asyncio.TimeoutError"
            if self.verbose_errors:
                traceback.print_exc()
            return 0.0
        except Exception as e:
            self.conn_err += 1
            self.last_error = f"{type(e).__name__}: {e}"
            if self.verbose_errors:
                traceback.print_exc()
            return 0.0
        finally:
            self._queue.put_nowait(env)

    async def _score_batch_async(
        self, tasks: list, commands: list[str]
    ) -> list[float]:
        return list(
            await asyncio.gather(
                *(self._score_one(t, c) for t, c in zip(tasks, commands))
            )
        )

    def score_batch(self, tasks: list, commands: list[str]) -> list[float]:
        """Parallel scoring; preserves input order."""
        assert self._loop is not None
        fut = asyncio.run_coroutine_threadsafe(
            self._score_batch_async(tasks, commands), self._loop
        )
        return fut.result()

    async def _rollout_one_async(
        self,
        task: Task,
        first_command: str,
        generate_next,           # sync callable (history) -> str
        max_steps: int,
    ) -> float:
        """Interactive multi-turn rollout on a single env session.

        Plays `first_command` as turn 1 (this is what TRL just generated),
        then for turns 2..max_steps asks `generate_next` for the next
        command given the running (cmd, output) history. Used by
        `score_batch_interactive` to give the trainer the same multi-step
        rollout semantics as eval-time `run_episode`.

        `generate_next` is sync because the policy lives on the main
        thread's GPU; we call it from this background-thread coroutine
        via `loop.run_in_executor(None, ...)` so it doesn't block the
        asyncio loop.
        """
        env = await self._queue.get()
        try:
            try:
                await asyncio.wait_for(env.reset(task=task), timeout=self.timeout_s)
                history: list[tuple[str, str]] = []
                reward = 0.0
                cmd = first_command
                loop = asyncio.get_running_loop()
                for turn in range(max_steps):
                    res = await asyncio.wait_for(
                        env.step(AwsRlAction(command=cmd)),
                        timeout=self.timeout_s,
                    )
                    reward = float(res.reward)
                    obs = getattr(res, "observation", None)
                    history.append((cmd, getattr(obs, "command_output", "") or ""))
                    done_flag = getattr(res, "done", False) or getattr(
                        obs, "task_achieved", False
                    )
                    if done_flag or turn == max_steps - 1:
                        break
                    # Hop to a worker thread for the (sync, GPU-bound) generation.
                    cmd = await loop.run_in_executor(
                        None, generate_next, tuple(history)
                    )
                self.success += 1
                return reward
            except ConnectionClosed as e:
                # Don't auto-retry mid-episode: a fresh reset would wipe
                # tracker state and any partial credit from earlier turns.
                self.last_error = f"{type(e).__name__}: {e} (mid-rollout)"
                if self.verbose_errors:
                    print(f"[reward] {self.last_error}")
                self.conn_err += 1
                return 0.0
        except asyncio.TimeoutError:
            self.timeout += 1
            self.last_error = "asyncio.TimeoutError"
            if self.verbose_errors:
                traceback.print_exc()
            return 0.0
        except Exception as e:
            self.conn_err += 1
            self.last_error = f"{type(e).__name__}: {e}"
            if self.verbose_errors:
                traceback.print_exc()
            return 0.0
        finally:
            self._queue.put_nowait(env)

    def score_batch_interactive(
        self,
        tasks: list,
        first_commands: list[str],
        generate_next_per_rollout: list,   # list[Callable[[history], str]]
        max_steps: int = 5,
    ) -> list[float]:
        """Parallel multi-turn scoring; preserves input order.

        Each rollout takes ONE command at a time: TRL's completion
        supplies turn 1, then `generate_next_per_rollout[i](history)`
        is called for each subsequent turn of rollout `i` until
        `done=True` or `max_steps` is reached. Up to `pool_size`
        rollouts run in parallel against the WS pool — but the caller
        is expected to serialise GPU-bound generates inside the
        closures themselves (one shared lock), since concurrent
        generates on a single accelerator only fight for the same
        memory.
        """
        assert self._loop is not None
        assert len(tasks) == len(first_commands) == len(generate_next_per_rollout)

        async def _gather():
            return list(await asyncio.gather(*(
                self._rollout_one_async(t, c, gen, max_steps)
                for t, c, gen in zip(
                    tasks, first_commands, generate_next_per_rollout,
                )
            )))

        fut = asyncio.run_coroutine_threadsafe(_gather(), self._loop)
        return fut.result()

    def drain_counters(self) -> dict:
        out = {
            "success": self.success,
            "timeout": self.timeout,
            "conn_err": self.conn_err,
            "reconnect": self.reconnect,
            "last_error": self.last_error,
        }
        self.success = self.timeout = self.conn_err = self.reconnect = 0
        self.last_error = None
        return out

    def close(self) -> None:
        if self._loop is None or not self._loop.is_running():
            return

        async def _close():
            await asyncio.gather(
                *(e.disconnect() for e in self._envs), return_exceptions=True
            )

        try:
            asyncio.run_coroutine_threadsafe(_close(), self._loop).result(timeout=30)
        except Exception:
            pass
        self._loop.call_soon_threadsafe(self._loop.stop)
        if self._thread is not None:
            self._thread.join(timeout=5)


def build_reward_funcs(env: EnvRewardClient,
                       task_map: dict[int, Task],
                       curriculum: Optional[Curriculum] = None,
                       model=None,
                       tokenizer=None,
                       max_rollout_steps: int = 5,
                       ) -> tuple[Callable, Callable, Callable]:
    """Return (env_reward, format_reward, length_reward).

    When `curriculum` is provided, env_reward calls curriculum.record_result()
    once per unique task_id in the batch — one record = one group = one
    curriculum episode. This is what makes the training loop curriculum-driven
    even while TRL owns the outer iteration.

    When `model` and `tokenizer` are provided, env_reward runs INTERACTIVE
    multi-turn rollouts: TRL's completion supplies the first command, then
    the policy is re-queried after each env step (with running command/output
    history) for up to `max_rollout_steps - 1` more turns. This is what lets
    multi-step (intermediate/advanced/expert) tasks actually achieve during
    training. Without `model`, env_reward falls back to single-step scoring.
    """
    def _text_of(c) -> str:
        return c if isinstance(c, str) else c[0]["content"]

    # Pre-build the multi-turn generate_next closure once. Only used when
    # `model` was supplied — single-step path skips it entirely.
    if model is not None and tokenizer is not None:
        import torch as _torch  # local import to avoid hard dep if unused
        _gen_lock = threading.Lock()  # GPU serialisation across rollouts

        def _generate_next_for_task(task: Task, history) -> str:
            """Sync per-turn generation: build prompt with running history,
            decode one command, return the extracted `aws ...` line.
            Runs on a worker thread off the event loop (see
            `_rollout_one_async`); the lock prevents concurrent rollouts
            from racing on the same GPU."""
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": f"TASK: {task.description}"},
            ]
            for cmd, out in list(history)[-4:]:
                messages.append({"role": "assistant", "content": cmd})
                messages.append({"role": "user",      "content": f"OUTPUT:\n{out[:400]}"})
            prompt_text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            with _gen_lock:
                inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
                with _torch.inference_mode():
                    ids = model.generate(
                        **inputs,
                        max_new_tokens=256,
                        do_sample=True, temperature=0.7, top_p=0.9,
                        pad_token_id=tokenizer.eos_token_id,
                    )
                text = tokenizer.decode(
                    ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True,
                )
            return extract_aws_command(text)
    else:
        _generate_next_for_task = None

    def env_reward(prompts, completions, task_id=None, **kw):
        tids = task_id if task_id is not None else kw["task_id"]
        tasks = [task_map[int(t)] for t in tids]
        cmds  = [extract_aws_command(_text_of(c)) for c in completions]

        if _generate_next_for_task is not None:
            # Interactive multi-turn: one task-bound generate_next closure
            # per rollout; env.step calls fan out across the WS pool, while
            # the shared `_gen_lock` inside the closure keeps generates
            # serial on the single GPU.
            gens = [
                (lambda hist, _t=t: _generate_next_for_task(_t, hist))
                for t in tasks
            ]
            rewards = env.score_batch_interactive(
                tasks, cmds, gens, max_steps=max_rollout_steps,
            )
        else:
            rewards = env.score_batch(tasks, cmds)

        # Group by task_id and feed each group back to the curriculum. TRL emits
        # G completions per prompt consecutively (all sharing one task_id), so
        # grouping recovers the GRPO semantics cleanly: one record per prompt,
        # achieved iff any rollout hit reward>=1.0, recorded reward = group mean.
        if curriculum is not None:
            by_tid: dict[int, list[float]] = defaultdict(list)
            for tid, r in zip(tids, rewards):
                by_tid[int(tid)].append(r)
            for tid, group in by_tid.items():
                curriculum.record_result(
                    task_map[tid],
                    achieved=any(r >= 1.0 for r in group),
                    reward=sum(group) / len(group),
                )
        return rewards

    def format_reward(prompts, completions, **kw):
        # Thinking is now allowed before the command, so the old
        # "first char must be aws " check would score 0 on every
        # well-formed <think>...</think> + aws ... output. Instead
        # require that *some* line of the completion starts with
        # "aws " — the same contract extract_aws_command() relies on.
        out = []
        for c in completions:
            txt = _text_of(c)
            has_cmd = any(
                line.strip().strip("`").strip().startswith("aws ")
                for line in txt.splitlines()
            )
            out.append(1.0 if has_cmd else 0.0)
        return out

    def length_reward(prompts, completions, **kw):
        # With <think>...</think> the completion can reasonably run to a
        # couple of thousand characters; the old 120-char cap would pin
        # length_reward to ~0 on every thinking output. Grade the extracted
        # command line for concision (commands themselves are still short)
        # and only mildly penalise extreme verbosity in the surrounding
        # reasoning so the model is nudged away from rambling.
        out = []
        for c in completions:
            txt = _text_of(c)
            cmd = extract_aws_command(txt)
            cmd_n = len(cmd)
            cmd_score = 1.0 if cmd_n <= 120 else max(
                0.0, 1.0 - (cmd_n - 120) / 280.0
            )
            total_n = len(txt)
            verbosity_score = 1.0 if total_n <= 2000 else max(
                0.0, 1.0 - (total_n - 2000) / 2000.0
            )
            out.append(0.5 * cmd_score + 0.5 * verbosity_score)
        return out

    return env_reward, format_reward, length_reward


# Re-run protection: if this cell has been run before, the old ENV_CLIENT's
# background thread is still holding 8 WebSocket sessions. Close it cleanly
# (sends {"type":"close"} so the server's /ws handler reaches its finally
# block and calls _destroy_session → releases the MiniStack slot). Without
# this, every re-run compounds the leak until the server hits 8/8 capacity.
try:
    _prev = ENV_CLIENT  # noqa: F821
except NameError:
    pass
else:
    log.info("Closing previous ENV_CLIENT to release its WS sessions…")
    try:
        _prev.close()
    except Exception as _e:
        log.warning("Ignored error during previous close: %r", _e)

# verbose_errors=True for the first run so the smoke test surfaces any WS /
# reset / step exception with a full traceback. Flip off after it passes.
ENV_CLIENT = EnvRewardClient(
    base_url=ENV_BASE_URL,
    pool_size=PIPE.env_pool_size,
    verbose_errors=True,
)

# Smoke test uses a non-curriculum client to avoid polluting any curriculum state.
# Task 0 ("List all S3 buckets") matches the `aws s3 ls` completion.
_smoke_task = TASK_MAP[0]
_smoke_env, _smoke_fmt, _smoke_len = build_reward_funcs(ENV_CLIENT, TASK_MAP, curriculum=None)
_smoke = _smoke_env(
    prompts=[None] * 2,
    completions=["aws s3 ls", "aws s3 ls"],
    task_id=[_smoke_task.task_id, _smoke_task.task_id],
)
log.info("Reward smoke test on task %s: %s", _smoke_task.task_id, _smoke)
log.info("Env counters: %s", ENV_CLIENT.drain_counters())
assert min(_smoke) > 0.5, "Reward smoke test failed — env or reward wiring broken"


14:53:16 | INFO    | grpo | Reward smoke test on task 0: [1.0, 1.0]
14:53:16 | INFO    | grpo | Env counters: {'success': 2, 'timeout': 0, 'conn_err': 0, 'reconnect': 0, 'last_error': None}


## 9 · Load the SFT adapter as the starting policy

We go through `PeftModel.from_pretrained(base, sft_adapter, is_trainable=True)` explicitly (rather than `FastLanguageModel.from_pretrained(adapter_repo)`) so there is no ambiguity about the adapter landing in trainable mode — GRPO must be able to update the LoRA weights.

This helper is also used later by the final-run and Optuna-trial paths, so it lives in its own cell.

In [11]:
import gc


def load_policy(spec: ModelSpec, trainable: bool = True):
    """Load base (4-bit) + SFT adapter.

    `trainable=True` gives a PeftModel with is_trainable=True — the correct
    mode for GRPO. `trainable=False` loads the adapter for inference only
    and invokes `FastLanguageModel.for_inference` for Unsloth's fused
    inference kernels (used during evaluation).
    """
    from unsloth import FastLanguageModel
    from peft import PeftModel

    base, tokenizer = FastLanguageModel.from_pretrained(
        model_name=spec.base_model,
        max_seq_length=spec.max_seq_length,
        load_in_4bit=True,
    )
    model = PeftModel.from_pretrained(base, spec.sft_adapter, is_trainable=trainable)
    if trainable:
        FastLanguageModel.for_training(model)
        # PeftModel.from_pretrained doesn't wire up the input-require-grads
        # forward hook on the embeddings. With Unsloth's gradient
        # checkpointing + 4-bit frozen base, that breaks the autograd path
        # between the loss and the trainable LoRA adapters — backward
        # errors with "element 0 of tensors does not require grad". The
        # hook marks the embedding output as requires_grad so gradients
        # can flow into the adapters. FastLanguageModel.get_peft_model
        # does this automatically; we don't use that path because we're
        # loading an existing SFT adapter, not creating a fresh one.
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
    else:
        FastLanguageModel.for_inference(model)
    return model, tokenizer


def free_model(model) -> None:
    """Release GPU memory held by a model + its optimizer state."""
    del model
    gc.collect()
    torch.cuda.empty_cache()


# Sanity: load + confirm LoRA params are trainable.
_probe_model, _probe_tok = load_policy(MODEL, trainable=True)
_trainable = [n for n, p in _probe_model.named_parameters() if p.requires_grad]
log.info("Loaded %s. Trainable params: %d tensors; sample: %s", MODEL.sft_adapter, len(_trainable), _trainable[:3])
assert any("lora" in n.lower() for n in _trainable), "No LoRA params marked trainable — load path is wrong"
free_model(_probe_model)
del _probe_tok
gc.collect(); torch.cuda.empty_cache()
log.info("Model load path verified.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

14:54:46 | INFO    | grpo | Loaded Sizzing/aws-rl-sft-qwen25coder3b-adapter. Trainable params: 288 tensors; sample: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight']
14:54:48 | INFO    | grpo | Model load path verified.


## 10 · Baseline eval — SFT adapter, single-step env reward

Single-pass eval on the val set. This is the "before" column of the headline comparison. The richer *multi-step* eval happens later; this one is only here to confirm the SFT-loaded model outputs sane commands against the env **before** we start GRPO.

Written as a small `evaluate_single_step` helper because GRPO's inner trial loop needs the same logic.

In [12]:
import json
import statistics as _stats
from dataclasses import asdict


@dataclass
class SingleStepMetrics:
    """One row of headline comparison numbers."""
    env_reward_mean:   float
    env_success_rate:  float   # fraction with reward >= 1.0
    format_pct:        float
    n:                 int

    def as_dict(self) -> dict:
        return asdict(self)


def evaluate_single_step(model, tokenizer, dataset, env: EnvRewardClient,
                         task_map: dict[int, Task],
                         max_new_tokens: int = 128) -> SingleStepMetrics:
    """Generate one command per prompt, score against the env, summarize."""
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)
    formats: list[float] = []
    tasks_to_score: list[Task] = []
    cmds_to_score:  list[str]  = []

    for row in dataset:
        prompt_text = tokenizer.apply_chat_template(
            row["prompt"], tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(
            ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True,
        )
        # Match the training-time format_reward contract: accept any
        # line starting with "aws " so a <think>...</think> prefix does
        # not score 0 for format.
        has_cmd = any(
            line.strip().strip("`").strip().startswith("aws ")
            for line in text.splitlines()
        )
        formats.append(1.0 if has_cmd else 0.0)
        tasks_to_score.append(task_map[int(row["task_id"])])
        cmds_to_score.append(extract_aws_command(text))

    # Score all env rewards in parallel across the 8 server slots
    rewards = env.score_batch(tasks_to_score, cmds_to_score)

    FastLanguageModel.for_training(model)

    return SingleStepMetrics(
        env_reward_mean=float(_stats.mean(rewards)),
        env_success_rate=sum(r >= 1.0 for r in rewards) / len(rewards),
        format_pct=float(_stats.mean(formats)),
        n=len(rewards),
    )


# Run the SFT-only baseline and persist it alongside Optuna + checkpoints
_baseline_model, _baseline_tok = load_policy(MODEL, trainable=False)
baseline_metrics = evaluate_single_step(
    _baseline_model, _baseline_tok, VAL_DS, ENV_CLIENT, TASK_MAP,
    max_new_tokens=TRAIN.max_completion_length,
)
(OUT_DIR / "baseline_single_step.json").write_text(json.dumps(baseline_metrics.as_dict(), indent=2))
free_model(_baseline_model); del _baseline_tok
gc.collect(); torch.cuda.empty_cache()

log.info("SFT baseline (single-step, val): %s", baseline_metrics.as_dict())


==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
14:56:31 | INFO    | grpo | SFT baseline (single-step, val): {'env_reward_mean': 0.9, 'env_success_rate': 0.85, 'format_pct': 1.0, 'n': 20}


In [13]:
import optuna
import time

log.info("Loading optuna and defining trial objective...")


def suggest_training_config(trial: optuna.Trial, defaults: TrainingConfig) -> TrainingConfig:
    """Return a mutated copy of `defaults` with Optuna-sampled hparams.

    One function, one diff — keeps the search space auditable.
    """
    return replace(
        defaults,
        learning_rate   = trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True),
        beta            = trial.suggest_float("beta", 0.0, 0.1),
        temperature     = trial.suggest_float("temperature", 0.7, 1.0),
    )


def trial_objective(trial: optuna.Trial) -> float:
    """Short curriculum-driven GRPO run + val eval. Returns mean env reward
    on the small Optuna task pool."""
    trial_cfg = suggest_training_config(trial, TRAIN)
    output_dir = str(OUT_DIR / f"optuna/trial-{trial.number}")
    log.info("[trial %d] starting | cfg=%s", trial.number, {
        "lr": trial_cfg.learning_rate, "beta": trial_cfg.beta,
        "temp": trial_cfg.temperature,
    })

    # Fresh curriculum per trial — otherwise mastery and tier progression bleed
    # across trials, making Optuna's hparam comparison unfair.
    trial_curriculum = Curriculum(tasks_dir=_tasks_dir)
    # Small fixed task pool (12 rows: warmup-2, beginner-2, intermediate-2,
    # advanced-3, expert-3). CurriculumTierSampler still cycles within whatever
    # tier the curriculum is on; with only 2-3 rows per tier the sampler
    # simply re-shuffles those indices each cycle.
    trial_train_ds = OPTUNA_DS
    trial_num_samples = int(
        PIPE.trial_max_steps
        * trial_cfg.per_device_train_batch_size
        * trial_cfg.gradient_accumulation_steps
        * 1.2
    )
    log.info("[trial %d] loading SFT policy (4-bit Qwen2.5-Coder-3B)...", trial.number)
    _t0 = time.time()
    model, tokenizer = load_policy(MODEL, trainable=True)
    log.info("[trial %d] policy loaded in %.1fs", trial.number, time.time() - _t0)

    trial_env_r, trial_fmt_r, trial_len_r = build_reward_funcs(
        ENV_CLIENT, TASK_MAP, trial_curriculum,
        model=model, tokenizer=tokenizer,
    )

    trainer = build_trainer(
        model, tokenizer,
        train_ds=trial_train_ds,
        eval_ds=OPTUNA_DS,
        reward_funcs=(trial_env_r, trial_fmt_r, trial_len_r),
        cfg=trial_cfg,
        output_dir=output_dir,
        run_name=f"optuna-trial-{trial.number}",
        use_fp16=RT.use_fp16, use_bf16=RT.use_bf16,
        max_steps=PIPE.trial_max_steps,
        save_strategy="no",
        curriculum=trial_curriculum,
        num_samples=trial_num_samples,
    )

    try:
        log.info("[trial %d] starting train() for %d steps...",
                 trial.number, PIPE.trial_max_steps)
        _t1 = time.time()
        trainer.train()
        log.info("[trial %d] train() finished in %.1fs", trial.number, time.time() - _t1)
        # Persist trainer_state.json for later plotting. save_strategy="no"
        # means TRL never writes checkpoints during training, so without
        # this call there'd be no on-disk log history for this trial once
        # the process exits.
        trial_out = Path(output_dir)
        trial_out.mkdir(parents=True, exist_ok=True)
        (trial_out / "trainer_state.json").write_text(
            json.dumps(
                {"log_history": trainer.state.log_history,
                 "global_step": trainer.state.global_step,
                 "trial_number": trial.number},
                indent=2, default=str,
            )
        )
        # Score the trial on the same small Optuna task pool.
        log.info("[trial %d] running single-step eval on %d tasks...",
                 trial.number, len(OPTUNA_DS))
        _t2 = time.time()
        metrics = evaluate_single_step(
            trainer.model, tokenizer, OPTUNA_DS, ENV_CLIENT, TASK_MAP,
            max_new_tokens=trial_cfg.max_completion_length,
        )
        log.info("[trial %d] eval finished in %.1fs", trial.number, time.time() - _t2)
        score = metrics.env_reward_mean
        # Also capture single-step eval metrics per trial for offline plotting.
        (trial_out / "single_step_metrics.json").write_text(
            json.dumps(metrics.as_dict(), indent=2)
        )
        log.info(
            "[trial %d] DONE | env_reward_mean=%.4f success=%.3f tier=%s graduated=%d",
            trial.number, score, metrics.env_success_rate,
            trial_curriculum.current_difficulty.value,
            len(trial_curriculum.get_stats().get("graduated_tasks", [])),
        )
    finally:
        free_model(trainer); free_model(model); del tokenizer
        gc.collect(); torch.cuda.empty_cache()

    trial.report(score, step=PIPE.trial_max_steps)
    if trial.should_prune():
        raise optuna.TrialPruned()
    return score


log.info("Curriculum-driven Optuna objective defined.")


14:56:31 | INFO    | grpo | Loading optuna and defining trial objective...
14:56:31 | INFO    | grpo | Curriculum-driven Optuna objective defined.


In [14]:
import random
import time
from collections import defaultdict
from collections.abc import Iterable

import torch
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback


class CurriculumTierSampler(torch.utils.data.Sampler[int]):
    """Reactive sampler: yields dataset indices from the curriculum's
    *current* tier, re-checked on every yield.

    The usual approach — pre-materialising a fixed dataset from the
    curriculum — freezes the training data to whatever tier the curriculum
    started in, because `curriculum.next_task()` never calls
    `record_result` and so `_maybe_promote` can't fire during the draw.
    Here the dataset is a *superset* of all tiers (see
    make_full_curriculum_dataset) and this sampler picks indices only
    from rows whose `difficulty` matches the curriculum's current tier.
    When `env_reward` promotes the curriculum mid-training, the very
    next yield flips onto the new tier's index pool.

    Implementation notes:
      * `__len__` is required for HF Trainer's max_steps math. We return
        `num_samples`, which the caller sizes as
        `max_steps * pdtbs * grad_accum * buffer`.
      * Index pools are built once at construction from the dataset's
        `difficulty` column; only the active pool + cursor are mutated
        during iteration, which keeps per-step overhead O(1).
      * If the curriculum's current tier has no tasks in the dataset
        (mis-configured YAML), we fall back to the union of all tiers so
        training doesn't deadlock.
    """

    def __init__(self, dataset, curriculum, num_samples: int) -> None:
        self.dataset = dataset
        self.curriculum = curriculum
        self.num_samples = int(num_samples)
        difficulties = list(dataset["difficulty"])
        self._pools: dict[str, list[int]] = defaultdict(list)
        for idx, diff in enumerate(difficulties):
            self._pools[diff].append(idx)
        self._all_indices = list(range(len(difficulties)))
        log.info(
            "CurriculumTierSampler: num_samples=%d, tier sizes=%s",
            self.num_samples,
            {k: len(v) for k, v in self._pools.items()},
        )

    def _pool_for(self, tier_value: str) -> list[int]:
        pool = self._pools.get(tier_value) or self._all_indices
        return list(pool)

    def __len__(self) -> int:
        return self.num_samples

    def __iter__(self):
        count = 0
        cur_tier: str | None = None
        pool: list[int] = []
        cursor = 0
        while count < self.num_samples:
            new_tier = self.curriculum.current_difficulty.value
            if new_tier != cur_tier:
                cur_tier = new_tier
                pool = self._pool_for(cur_tier)
                random.shuffle(pool)
                cursor = 0
            if cursor >= len(pool):
                random.shuffle(pool)
                cursor = 0
            yield pool[cursor]
            cursor += 1
            count += 1


class CurriculumPromotionCallback(TrainerCallback):
    """Observability: log when the curriculum promotes."""

    def __init__(self, curriculum) -> None:
        self.curriculum = curriculum
        self._last_tier: str | None = None

    def on_step_end(self, args, state, control, **kw):
        tier = self.curriculum.current_difficulty.value
        if tier != self._last_tier:
            if self._last_tier is not None:
                log.info(
                    "[CurriculumPromotionCallback] tier promoted %s -> %s at step %d",
                    self._last_tier, tier, state.global_step,
                )
            self._last_tier = tier


class EnvHealthCallback(TrainerCallback):
    """Log env health counters + drain them every N steps.

    Also provides an early-warning bail-out: if every scoring call in a
    window came back as timeout/conn_err, the hosted env is probably down
    and we want to stop training rather than polluting the adapter with
    zero-reward updates.
    """

    def __init__(self, env_client: EnvRewardClient, probe_every: int = 50,
                 fail_threshold: int = 32) -> None:
        self.env = env_client
        self.probe_every = probe_every
        self.fail_threshold = fail_threshold

    def on_log(self, args, state, control, logs=None, **kw):
        if state.global_step == 0 or state.global_step % self.probe_every != 0:
            return
        counters = self.env.drain_counters()
        log.info("[env counters] step=%d %s", state.global_step, counters)
        if counters["timeout"] + counters["conn_err"] >= self.fail_threshold:
            log.error("[EnvHealthCallback] %s at step %d — stopping training.",
                      counters, state.global_step)
            control.should_training_stop = True


class ProgressLogCallback(TrainerCallback):
    """Mirror TRL's scalar logs (loss, reward, kl, ...) to the Python logger.

    TRL writes its scalars via `Trainer.log`, which goes to `state.log_history`
    and to whichever integrations are listed in `report_to`. With
    `report_to="none"` there is no on-screen feedback during training — just
    a tqdm bar that Jupyter sometimes hides. This callback hooks `on_log` and
    forwards the scalar dict through `log.info` every step, plus a heartbeat
    line on `on_step_end` every `heartbeat_every` steps so you always see
    *something* even when TRL hasn't emitted a scalar yet (e.g. the opening
    generation phase before the first optimizer step).
    """

    def __init__(self, heartbeat_every: int = 1, run_label: str = "") -> None:
        self.heartbeat_every = max(1, int(heartbeat_every))
        self.run_label = run_label
        self._t_start: float | None = None
        self._t_last: float | None = None

    def on_train_begin(self, args, state, control, **kw):
        self._t_start = time.time()
        self._t_last = self._t_start
        log.info("%s train_begin | max_steps=%s", self._tag(), args.max_steps)

    def on_step_end(self, args, state, control, **kw):
        if state.global_step % self.heartbeat_every != 0:
            return
        now = time.time()
        dt = now - (self._t_last or now)
        self._t_last = now
        log.info("%s step %d/%s (+%.1fs)",
                 self._tag(), state.global_step, args.max_steps, dt)

    def on_log(self, args, state, control, logs=None, **kw):
        if not logs:
            return
        # Drop noisy / non-scalar fields and round for legibility.
        scalars = {k: (round(v, 4) if isinstance(v, float) else v)
                   for k, v in logs.items()
                   if isinstance(v, (int, float))}
        if scalars:
            log.info("%s log step=%d %s",
                     self._tag(), state.global_step, scalars)

    def on_train_end(self, args, state, control, **kw):
        elapsed = time.time() - (self._t_start or time.time())
        log.info("%s train_end | global_step=%d elapsed=%.1fs",
                 self._tag(), state.global_step, elapsed)

    def _tag(self) -> str:
        return f"[{self.run_label}]" if self.run_label else "[train]"


def build_trainer(model, tokenizer, train_ds, eval_ds,
                  reward_funcs: Iterable[Callable],
                  cfg: TrainingConfig, *,
                  output_dir: str, run_name: str,
                  use_fp16: bool, use_bf16: bool,
                  max_steps: int | None = None,
                  save_strategy: str = "steps",
                  eval_strategy: str = "steps",
                  curriculum=None,
                  num_samples: int | None = None) -> GRPOTrainer:
    """Assemble a GRPOTrainer from a typed TrainingConfig.

    When `curriculum` is supplied, the trainer's train sampler is
    replaced with a CurriculumTierSampler that yields dataset indices
    from the curriculum's current tier — reactive to promotion. The
    dataset passed in as `train_ds` must carry a `difficulty` column
    (use make_full_curriculum_dataset). `num_samples` sizes the
    sampler; callers typically set it to
    `max_steps * per_device_train_batch_size * gradient_accumulation_steps * 1.2`.
    """
    args = GRPOConfig(
        output_dir=output_dir, run_name=run_name,
        num_generations=cfg.num_generations, beta=cfg.beta,
        temperature=cfg.temperature, top_p=cfg.top_p,
        max_prompt_length=cfg.max_prompt_length,
        max_completion_length=cfg.max_completion_length,
        learning_rate=cfg.learning_rate, lr_scheduler_type="cosine",
        optim="adamw_8bit", weight_decay=0.0, max_grad_norm=1.0,
        warmup_ratio=cfg.warmup_ratio,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        # TRL's GRPOConfig asserts per_device_eval_batch_size * num_processes is
        # divisible by num_generations (one eval prompt produces G completions;
        # anything smaller can't form a group). Defaulting it to num_generations
        # is the smallest value that satisfies the check on a single-process
        # setup — matches how GRPOTrainer batches eval internally.
        per_device_eval_batch_size=cfg.num_generations,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        num_train_epochs=cfg.num_train_epochs,
        max_steps=(max_steps if max_steps is not None else -1),
        fp16=use_fp16, bf16=use_bf16,
        # Mid-training eval is disabled for GRPO: TRL's prediction_step
        # calls _generate_and_score_completions, which reshapes rewards
        # as (-1, num_generations). At eval time the effective per-prompt
        # completion count can differ from num_generations, so the view
        # errors with "shape '[-1, G]' is invalid for input of size N".
        # The notebook runs its own before/after eval via evaluate_single_step,
        # so we lose nothing by skipping TRL's eval loop here.
        eval_strategy=eval_strategy, eval_steps=cfg.eval_steps,
        save_strategy=save_strategy, save_steps=cfg.save_steps,
        save_total_limit=cfg.save_total_limit,
        logging_steps=1, report_to="none", seed=cfg.seed,
        remove_unused_columns=False,   # CRITICAL: preserves task_id for reward_fns
        disable_tqdm=False,
    )
    callbacks = [
        EnvHealthCallback(ENV_CLIENT),
        ProgressLogCallback(heartbeat_every=1, run_label=run_name),
    ]
    if curriculum is not None:
        callbacks.append(CurriculumPromotionCallback(curriculum))

    trainer = GRPOTrainer(
        model=model, processing_class=tokenizer,
        reward_funcs=list(reward_funcs),
        reward_weights=[1.0, 0.15, 0.05],
        args=args, train_dataset=train_ds, eval_dataset=eval_ds,
        callbacks=callbacks,
    )

    if curriculum is not None:
        if num_samples is None:
            raise ValueError("num_samples is required when curriculum is set")
        if "difficulty" not in train_ds.column_names:
            raise ValueError(
                "curriculum-driven training needs a `difficulty` column on "
                "train_ds — use make_full_curriculum_dataset()."
            )
        _sampler = CurriculumTierSampler(
            dataset=train_ds, curriculum=curriculum, num_samples=num_samples,
        )
        # Monkey-patch the trainer's sampler factory so the DataLoader,
        # which is built lazily inside trainer.train(), pulls from our
        # tier-reactive sampler instead of the default RandomSampler.
        # Using MethodType keeps `self` binding correct on the bound call.
        import types
        def _curriculum_train_sampler(self):
            return _sampler
        trainer._get_train_sampler = types.MethodType(
            _curriculum_train_sampler, trainer,
        )

    return trainer


log.info("GRPOTrainer factory ready.")


14:56:32 | INFO    | grpo | GRPOTrainer factory ready.


## 12 · Optuna hyperparameter search

Six short trials (30 GRPO steps each, 80-row training subset). Each trial returns the mean **env reward on the held-out val set** — the one metric we actually want to maximize.

- **Resumable**: sqlite storage + `load_if_exists=True`. A dropped Colab session picks up where it left off.
- **Pruned**: `MedianPruner` — kill trials that are trending below the median after 10 steps.
- **Search space** chosen to bracket the GRPO-on-3B defaults reported in the DeepSeek-Math paper.

In [15]:
# Reclaim VRAM before starting trials. If the user ran the final-run cell
# first (e.g. because optuna_best.json didn't exist on a prior session),
# `final_model` / `final_trainer` / `_probe_model` may still be resident
# on the GPU. Each trial loads a fresh 4-bit Qwen (~6 GB) via
# load_policy, and on a 14.5 GB T4 the second model won't fit alongside
# leftovers, so BitsAndBytes falls back to CPU offload and errors out
# ("Some modules are dispatched on the CPU or the disk").
for _name in ("final_trainer", "final_model", "final_tok", "_probe_model", "_probe_tok", "_baseline_model", "_baseline_tok"):
    if _name in globals():
        try:
            _obj = globals().pop(_name)
            del _obj
        except Exception:
            pass
gc.collect()
try:
    torch.cuda.empty_cache()
except Exception:
    pass

STUDY_DB = OUT_DIR / "optuna.db"
STUDY_NAME = "aws-rl-grpo-search"

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=TRAIN.seed),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=10),
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
)

completed = sum(1 for t in study.trials if t.state.name == "COMPLETE")
remaining = max(0, PIPE.n_trials - completed)
print(f"Optuna study '{STUDY_NAME}': {completed} completed, {remaining} remaining.")

if remaining > 0:
    study.optimize(trial_objective, n_trials=remaining)

best_cfg = replace(TRAIN, **{
    k: v for k, v in study.best_params.items() if k in TrainingConfig.__dataclass_fields__
})
(OUT_DIR / "optuna_best.json").write_text(json.dumps({
    "best_value": study.best_value,
    "best_params": study.best_params,
    "resolved_config": asdict(best_cfg),
}, indent=2))

print(f"Best trial env_reward_mean = {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-04-25 14:56:34,668] A new study created in RDB with name: aws-rl-grpo-search


Optuna study 'aws-rl-grpo-search': 0 completed, 4 remaining.
14:56:34 | INFO    | grpo | [trial 0] starting | cfg={'lr': 4.328450221293881e-06, 'beta': 0.09507143064099162, 'temp': 0.9195981825434215}
14:56:34 | INFO    | server.services.curriculum | Loaded 25 warmup tasks total
14:56:34 | INFO    | server.services.curriculum | Curriculum initialised — starting at warmup with 25 tasks
14:56:34 | INFO    | grpo | [trial 0] loading SFT policy (4-bit Qwen2.5-Coder-3B)...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
14:56:52 | INFO    | grpo | [trial 0] policy loaded in 17.9s
14:56:52 | INFO

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12 | Num Epochs = 5 | Total steps = 10
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)


14:56:55 | INFO    | grpo | [optuna-trial-0] train_begin | max_steps=10
Unsloth: Will smartly offload gradients to save VRAM!
14:59:00 | INFO    | server.services.curriculum | Episode 1: task=33 difficulty=warmup achieved=True tier_rate=1.00
14:59:00 | INFO    | server.services.curriculum | Episode 2: task=37 difficulty=warmup achieved=True tier_rate=1.00
14:59:43 | INFO    | grpo | [optuna-trial-0] step 1/10 (+168.1s)


Step,Training Loss,Validation Loss


...

15:55:01 | INFO    | grpo | [optuna-trial-3] train_begin | max_steps=10
Unsloth: Will smartly offload gradients to save VRAM!
15:55:26 | INFO    | server.services.curriculum | Episode 1: task=33 difficulty=warmup achieved=True tier_rate=1.00
15:55:26 | INFO    | server.services.curriculum | Episode 2: task=37 difficulty=warmup achieved=True tier_rate=1.00
15:55:35 | INFO    | grpo | [optuna-trial-3] step 1/10 (+34.0s)


Step,Training Loss,Validation Loss


15:55:35 | INFO    | grpo | [optuna-trial-3] log step=1 {'loss': -0.062, 'grad_norm': 0.3466, 'learning_rate': 0.0, 'num_tokens': 3349.0, 'completions/mean_length': 43.3125, 'completions/min_length': 24.0, 'completions/max_length': 61.0, 'completions/clipped_ratio': 0.0, 'completions/mean_terminated_length': 43.3125, 'completions/min_terminated_length': 24.0, 'completions/max_terminated_length': 61.0, 'rewards/env_reward/mean': 0.8125, 'rewards/env_reward/std': 0.4031, 'rewards/format_reward/mean': 1.0, 'rewards/format_reward/std': 0.0, 'rewards/length_reward/mean': 1.0, 'rewards/length_reward/std': 0.0, 'reward': 2.8125, 'reward_std': 0.4082, 'frac_reward_zero_std': 0.0, 'completion_length': 43.3125, 'kl': 0.1133, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'epoch': 0.6667}
15:56:04 | INFO    | server.services.curriculum | Loaded 25 beginner tasks total
15:56:04 | INFO    | server.servic

[I 2026-04-25 16:12:04,437] Trial 3 finished with value: 0.552 and parameters: {'learning_rate': 1.5958573588141284e-05, 'beta': 0.0020584494295802446, 'temperature': 0.9909729556485982}. Best is trial 3 with value: 0.552.


Best trial env_reward_mean = 0.5520
Best params: {'learning_rate': 1.5958573588141284e-05, 'beta': 0.0020584494295802446, 'temperature': 0.9909729556485982}


In [16]:
import matplotlib.pyplot as plt
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_parallel_coordinate,
    plot_param_importances,
)

GRAPHS_DIR = OUT_DIR / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

for name, fn in [
    ("history",     plot_optimization_history),
    ("parallel",    plot_parallel_coordinate),
    ("importances", plot_param_importances),
]:
    try:
        ax = fn(study)
        fig = ax.figure
        out_png = GRAPHS_DIR / f"00_optuna_{name}.png"
        fig.savefig(out_png, dpi=150, bbox_inches="tight")
        plt.close(fig)
        log.info("Saved %s", out_png)
    except Exception as e:
        log.warning("Optuna %s plot skipped: %s", name, e)


/tmp/ipykernel_1159/2728918386.py:17: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = fn(study)


16:12:05 | INFO    | grpo | Saved /content/out/graphs/00_optuna_history.png


/tmp/ipykernel_1159/2728918386.py:17: ExperimentalWarning: optuna.visualization.matplotlib._parallel_coordinate.plot_parallel_coordinate is experimental (supported from v2.2.0). The interface can change in the future.
  ax = fn(study)


16:12:05 | INFO    | grpo | Saved /content/out/graphs/00_optuna_parallel.png


/tmp/ipykernel_1159/2728918386.py:17: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  ax = fn(study)


16:12:06 | INFO    | grpo | Saved /content/out/graphs/00_optuna_importances.png


## 14 · Final GRPO run — best hparams, full data, checkpointed

Uses the Optuna winner on the full 1-epoch training set with periodic checkpoints (`save_steps=25`, `save_total_limit=3`). `CheckpointManager.latest()` auto-detects a prior checkpoint so re-running this cell after a disconnect resumes safely.

Wandb run `final-grpo` gets a stable run id so resumed training stitches into the same panel instead of creating a fresh chart.

In [17]:
import json
from dataclasses import asdict


class CheckpointManager:
    """Find the latest `checkpoint-N/` under `root` for safe resume."""

    def __init__(self, root: Path) -> None:
        self.root = root

    def latest(self) -> str | None:
        if not self.root.exists():
            return None
        ckpts = sorted(
            (d for d in self.root.glob("checkpoint-*") if d.is_dir()),
            key=lambda d: int(d.name.split("-")[-1]),
        )
        return str(ckpts[-1]) if ckpts else None


FINAL_RUN_DIR = OUT_DIR / "final_grpo"
ckpt_mgr = CheckpointManager(FINAL_RUN_DIR)
resume_from = ckpt_mgr.latest()

# Make the final-run cell re-runnable after a kernel restart: if `best_cfg` is
# not in the namespace (Optuna didn't run this session), recover it from the
# persisted optuna_best.json; if that's missing too, fall back to TRAIN
# defaults so the run still launches on the sane baseline hparams.
try:
    best_cfg
except NameError:
    _best_path = OUT_DIR / "optuna_best.json"
    if _best_path.exists():
        _resolved = json.loads(_best_path.read_text())["resolved_config"]
        best_cfg = replace(TRAIN, **{
            k: v for k, v in _resolved.items()
            if k in TrainingConfig.__dataclass_fields__
        })
        log.info("Recovered best_cfg from %s", _best_path)
    else:
        best_cfg = TRAIN
        log.info("No Optuna results found; using TRAIN defaults for best_cfg")

# Master curriculum for the final run — single instance that progresses
# through tiers as the agent demonstrates mastery. Lives for the whole run.
FINAL_CURRICULUM = Curriculum(tasks_dir=_tasks_dir)
# Superset dataset (all tiers). The CurriculumTierSampler (wired in
# build_trainer) filters indices by FINAL_CURRICULUM.current_difficulty
# on every yield, so promotion during training takes effect on the very
# next batch instead of being frozen out by an up-front materialisation.
FINAL_TRAIN_DS = make_full_curriculum_dataset(_tasks_dir)
_final_num_samples = int(
    PIPE.final_max_steps
    * best_cfg.per_device_train_batch_size
    * best_cfg.gradient_accumulation_steps
    * 1.2
)
final_model, final_tok = load_policy(MODEL, trainable=True)

final_env_r, final_fmt_r, final_len_r = build_reward_funcs(
    ENV_CLIENT, TASK_MAP, FINAL_CURRICULUM,
    model=final_model, tokenizer=final_tok,
)

final_trainer = build_trainer(
    final_model, final_tok,
    train_ds=FINAL_TRAIN_DS,
    eval_ds=VAL_DS,
    reward_funcs=(final_env_r, final_fmt_r, final_len_r),
    cfg=best_cfg,
    output_dir=str(FINAL_RUN_DIR),
    run_name="final-grpo",
    use_fp16=RT.use_fp16, use_bf16=RT.use_bf16,
    max_steps=PIPE.final_max_steps,
    save_strategy="steps",
    # Skip TRL's mid-training eval — its GRPO prediction_step reshapes
    # rewards as (-1, num_generations) which blows up when eval generates
    # a different completion count per prompt than training. We do full
    # env-reward eval outside TRL via evaluate_single_step anyway.
    eval_strategy="no",
    curriculum=FINAL_CURRICULUM,
    num_samples=_final_num_samples,
)

if resume_from:
    log.info("Resuming GRPO from %s", resume_from)
final_trainer.train(resume_from_checkpoint=resume_from)

# Log final curriculum stats so we can see tier progression / mastery counts
final_stats = FINAL_CURRICULUM.get_stats()
log.info("Final curriculum stats: %s", final_stats)

# Persist the final LoRA adapter locally before anything else touches VRAM
adapter_local = OUT_DIR / "grpo_adapter"
final_trainer.model.save_pretrained(str(adapter_local))
final_tok.save_pretrained(str(adapter_local))
log.info("Saved GRPO adapter to %s", adapter_local)


16:12:28 | INFO    | server.services.curriculum | Loaded 25 warmup tasks total
16:12:28 | INFO    | server.services.curriculum | Curriculum initialised — starting at warmup with 25 tasks
16:12:28 | INFO    | server.services.curriculum | Loaded 25 warmup tasks total
16:12:28 | INFO    | server.services.curriculum | Loaded 25 beginner tasks total
16:12:28 | INFO    | server.services.curriculum | Loaded 25 intermediate tasks total
16:12:28 | INFO    | server.services.curriculum | Loaded 25 advanced tasks total
16:12:28 | INFO    | server.services.curriculum | Loaded 9 supplementary expert tasks from drift.yaml
16:12:28 | INFO    | server.services.curriculum | Loaded 33 expert tasks total
16:12:28 | INFO    | grpo | Full curriculum dataset: 133 rows across tiers ['advanced', 'beginner', 'expert', 'intermediate', 'warmup']
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Tor

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 133 | Num Epochs = 6 | Total steps = 35
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)


16:12:52 | INFO    | grpo | [final-grpo] train_begin | max_steps=35
Unsloth: Will smartly offload gradients to save VRAM!
16:13:18 | INFO    | server.services.curriculum | Episode 1: task=37 difficulty=warmup achieved=True tier_rate=1.00
16:13:18 | INFO    | server.services.curriculum | Episode 2: task=33 difficulty=warmup achieved=False tier_rate=0.46
16:13:18 | INFO    | server.services.curriculum | Episode 3: task=30 difficulty=warmup achieved=True tier_rate=0.67
16:13:18 | INFO    | server.services.curriculum | Episode 4: task=40 difficulty=warmup achieved=True tier_rate=0.77
16:13:18 | INFO    | server.services.curriculum | Episode 5: task=39 difficulty=warmup achieved=False tier_rate=0.56
16:13:18 | INFO    | server.services.curriculum | Episode 6: task=27 difficulty=warmup achieved=False tier_rate=0.43
16:13:18 | INFO    | server.services.curriculum | Episode 7: task=5 difficulty=warmup achieved=True tier_rate=0.55
16:13:18 | INFO    | server.services.curriculum | Loaded 25 begi

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / env_reward / mean,rewards / env_reward / std,rewards / format_reward / mean,rewards / format_reward / std,rewards / length_reward / mean,rewards / length_reward / std
1,0.031300,2.812500,0.258775,37.625000,21.000000,69.000000,0.000000,37.625000,21.000000,69.000000,0.160042,0.812500,0.403113,1.000000,0.000000,1.000000,0.000000
2,0.001200,2.937500,0.176777,44.812500,17.000000,75.000000,0.000000,44.812500,17.000000,75.000000,0.141318,0.937500,0.250000,1.000000,0.000000,1.000000,0.000000
3,-0.052300,2.218824,0.336094,109.687500,62.000000,242.000000,0.000000,109.687500,62.000000,242.000000,0.176999,0.311905,0.275233,1.000000,0.000000,0.906920,0.159451
4,0.037300,2.217083,0.357512,80.812500,40.000000,144.000000,0.000000,80.812500,40.000000,144.000000,0.138093,0.276458,0.319758,1.000000,0.000000,0.940625,0.116300
5,0.057100,2.230664,0.419722,67.250000,16.000000,128.000000,0.000000,67.250000,16.000000,128.000000,0.194109,0.281333,0.370841,1.000000,0.000000,0.949330,0.106090
6,0.357500,2.306159,0.492075,89.375000,31.000000,165.000000,0.000000,89.375000,31.000000,165.000000,0.111130,0.379708,0.410870,1.000000,0.000000,0.926451,0.142032
7,0.142100,2.151121,0.471804,97.125000,47.000000,235.000000,0.000000,97.125000,47.000000,235.000000,0.151401,0.255250,0.376225,1.000000,0.000000,0.895871,0.149889
8,-0.187200,2.124732,0.502367,90.187500,30.000000,191.000000,0.000000,90.187500,30.000000,191.000000,0.128590,0.247500,0.385668,1.000000,0.000000,0.877232,0.184127
9,0.313800,2.191510,0.337312,95.375000,28.000000,304.000000,0.000000,95.375000,28.000000,304.000000,0.114644,0.257917,0.271222,1.000000,0.000000,0.933594,0.140376
10,0.086600,2.191076,0.368338,90.625000,43.000000,164.000000,0.000000,90.625000,43.000000,164.000000,0.218185,0.255250,0.345282,1.000000,0.000000,0.935826,0.107090


16:13:25 | INFO    | grpo | [final-grpo] log step=1 {'loss': 0.0313, 'grad_norm': 0.5735, 'learning_rate': 0.0, 'num_tokens': 3235.0, 'completions/mean_length': 37.625, 'completions/min_length': 21.0, 'completions/max_length': 69.0, 'completions/clipped_ratio': 0.0, 'completions/mean_terminated_length': 37.625, 'completions/min_terminated_length': 21.0, 'completions/max_terminated_length': 69.0, 'rewards/env_reward/mean': 0.8125, 'rewards/env_reward/std': 0.4031, 'rewards/format_reward/mean': 1.0, 'rewards/format_reward/std': 0.0, 'rewards/length_reward/mean': 1.0, 'rewards/length_reward/std': 0.0, 'reward': 2.8125, 'reward_std': 0.2588, 'frac_reward_zero_std': 0.5, 'completion_length': 37.625, 'kl': 0.16, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'epoch': 0.1905}
16:13:51 | INFO    | server.services.curriculum | Loaded 9 supplementary expert tasks from drift.yaml
16:13:51 | INFO    | s

## 15 · Multi-step evaluation harness

Training was single-step for TRL compatibility; *evaluation* runs full episodes so we can measure:

- per-tier task success
- hints used per solved task
- recovery rate after chaos injection
- drift repair rate
- steps to solve
- rollback count (safety)
- generalization gap (reserve vs train-held-out)

The pattern mirrors [`scripts/grpo_train.py`](https://github.com/UdayKiranPadhy/aws-rl-env/blob/master/scripts/grpo_train.py)'s `run_single_rollout`, but uses `GrpoPool` for 8-way concurrent rollouts so a 100-task eval finishes in ~7 minutes.

In [18]:
import asyncio
from collections import defaultdict
from models import AwsRlAction
from client import AwsRlEnv
from scripts.grpo_pool import GrpoPool


# Multi-step eval uses a richer prompt that includes command history.
# Separate from the single-step SYSTEM_PROMPT to avoid shadowing it.
EVAL_SYSTEM_PROMPT = (
    "You are an expert AWS SRE agent. You operate a simulated AWS cloud by "
    "emitting one AWS CLI command at a time. You will see the task description "
    "and the most recent command output.\n\n"
    "First reason about your next move inside a <think>...</think> block: use "
    "the latest OUTPUT to decide what to do next, pick the AWS service and "
    "subcommand, and list the arguments you need. Keep the reasoning brief.\n\n"
    "After </think>, on a NEW LINE, output EXACTLY ONE AWS CLI command starting "
    "with 'aws '. The command line must contain only the command \u2014 no "
    "markdown, no backticks, no quotes around it, and no trailing commentary."
)


def build_multi_step_prompt(tokenizer, task: Task,
                             history: list[tuple[str, str]]) -> str:
    """Assemble chat-style prompt including the last few (cmd, output) turns."""
    messages = [
        {"role": "system", "content": EVAL_SYSTEM_PROMPT},
        {"role": "user",   "content": f"TASK: {task.description}"},
    ]
    for cmd, out in history[-4:]:   # last 4 turns fit in 512 tokens
        messages.append({"role": "assistant", "content": cmd})
        messages.append({"role": "user",      "content": f"OUTPUT:\n{out[:400]}"})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )


@dataclass
class EpisodeResult:
    task_id: int
    tier: str
    is_drift: bool
    achieved: bool
    terminal_reward: float
    steps_taken: int
    hints_used: int
    chaos_occurred: bool
    command_failures: int


async def run_episode(env: AwsRlEnv, model, tokenizer,
                      task: Task, drift_ids: set[int],
                      max_steps: int = 15,
                      # Bumped 96 → 512 so per-step generation has room
                      # for a <think>…</think> block plus the command.
                      max_new_tokens: int = 512) -> EpisodeResult:
    """Roll one episode against one env session. Sampling temperature is low
    to reflect deployment behaviour rather than training-time exploration."""
    device = next(model.parameters()).device
    res = await env.reset(task=task)
    history: list[tuple[str, str]] = []
    steps_taken = 0
    command_failures = 0
    terminal_reward = 0.0
    achieved = False

    for _ in range(max_steps):
        steps_taken += 1
        prompt = build_multi_step_prompt(tokenizer, task, history)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.4,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(
            ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True,
        )
        cmd = extract_aws_command(text)
        res = await env.step(AwsRlAction(command=cmd))
        terminal_reward = float(res.reward)
        obs = res.observation
        if not obs.command_success:
            command_failures += 1
        history.append((cmd, obs.command_output or ""))
        if obs.task_achieved:
            achieved = True
        if res.done:
            break

    # One final /state poll for chaos flag — TrackerState doesn't expose
    # rollback counts, so we skip that metric rather than report zeros.
    try:
        state = await env.state()
        chaos = bool(getattr(state, "chaos_occurred", False))
    except Exception:
        chaos = False

    return EpisodeResult(
        task_id=int(task.task_id),
        tier=task.difficulty.value,
        is_drift=int(task.task_id) in drift_ids,
        achieved=achieved,
        terminal_reward=terminal_reward,
        steps_taken=steps_taken,
        hints_used=int(getattr(res.observation, "hints_used", 0) or 0),
        chaos_occurred=chaos,
        command_failures=command_failures,
    )


log.info("run_episode defined.")

18:07:19 | INFO    | grpo | run_episode defined.


In [28]:
@dataclass
class RichMetrics:
    """All the metrics the hackathon judges care about."""
    success_by_tier:       dict
    reward_by_tier:        dict
    overall_success_rate:  float
    overall_reward_mean:   float
    hints_per_solved:      float
    recovery_rate:         float
    drift_repair_rate:     float
    steps_to_solve:        float
    destructive_fail_rate: float
    n_episodes:            int

    def as_dict(self) -> dict:
        return asdict(self)


def summarize_episodes(results: list[EpisodeResult]) -> RichMetrics:
    """Aggregate per-tier and overall stats from a list of EpisodeResults.

    Drift detection uses the per-result `is_drift` flag (set from DRIFT_TASK_IDS)
    rather than a tier string — drift tasks live inside the EXPERT tier files.
    """
    by_tier: dict[str, list[EpisodeResult]] = defaultdict(list)
    for r in results:
        by_tier[r.tier].append(r)

    success_by_tier = {tier: sum(r.achieved for r in xs) / max(1, len(xs))
                       for tier, xs in by_tier.items()}
    reward_by_tier  = {tier: (sum(r.terminal_reward for r in xs) / max(1, len(xs)))
                       for tier, xs in by_tier.items()}

    solved = [r for r in results if r.achieved]
    chaos_episodes = [r for r in results if r.chaos_occurred]
    drift_episodes = [r for r in results if r.is_drift]

    return RichMetrics(
        success_by_tier=success_by_tier,
        reward_by_tier=reward_by_tier,
        overall_success_rate=len(solved) / max(1, len(results)),
        overall_reward_mean=sum(r.terminal_reward for r in results) / max(1, len(results)),
        hints_per_solved=(sum(r.hints_used for r in solved) / len(solved)) if solved else 0.0,
        recovery_rate=(sum(r.achieved for r in chaos_episodes) / len(chaos_episodes))
                      if chaos_episodes else 0.0,
        drift_repair_rate=(sum(r.achieved for r in drift_episodes) / len(drift_episodes))
                          if drift_episodes else 0.0,
        steps_to_solve=(sum(r.steps_taken for r in solved) / len(solved)) if solved else 0.0,
        destructive_fail_rate=sum(r.command_failures > 0 for r in results) / max(1, len(results)),
        n_episodes=len(results),
    )


async def evaluate_multi_step(base_url: str, task_ids: list[int],
                              task_map: dict[int, Task],
                              drift_ids: set[int],
                              model, tokenizer,
                              pool_size: int = 8,
                              max_steps: int = 15) -> RichMetrics:
    """Run one episode per task_id across `pool_size` concurrent env sessions.

    Opens a fresh GrpoPool per chunk. Reusing a single pool across chunks
    fails because the env server / Cloudflare tunnel closes the WebSocket
    (code 1000) once an episode terminates or the connection idles during
    a long generate() call; the next chunk's reset() then raises
    ConnectionClosedOK on a dead socket. Per-chunk pools cost N WebSocket
    handshakes per chunk but keep the eval correct.
    """
    results: list[EpisodeResult] = []
    chunks = [task_ids[i:i + pool_size]
              for i in range(0, len(task_ids), pool_size)]
    log.info("multi-step eval: %d tasks in %d chunks (pool_size=%d)",
             len(task_ids), len(chunks), pool_size)
    for ci, chunk in enumerate(chunks, 1):
        async with GrpoPool(base_url=base_url, size=len(chunk)) as pool:
            coros = [run_episode(env, model, tokenizer, task_map[tid],
                                  drift_ids, max_steps=max_steps)
                     for env, tid in zip(pool.envs, chunk)]
            chunk_results = await asyncio.gather(*coros, return_exceptions=True)
        ok = sum(1 for r in chunk_results if isinstance(r, EpisodeResult))
        log.info("chunk %d/%d: %d/%d episodes ok",
                 ci, len(chunks), ok, len(chunk))
        for r in chunk_results:
            if isinstance(r, EpisodeResult):
                results.append(r)
            else:
                log.warning("[eval] episode raised %s: %s",
                            type(r).__name__, r)
    if not results:
        log.error("multi-step eval produced 0 results — env connection problem?")
    return summarize_episodes(results)


def select_eval_task_ids(reserve_ds, drift_ids: set[int], cap: int) -> list[int]:
    """Task ids for the multi-step eval: reserve split + every drift task."""
    tids = [int(r["task_id"]) for r in reserve_ds][:cap] if reserve_ds else []
    for did in drift_ids:
        if did not in tids:
            tids.append(did)
    return tids


log.info("evaluate_multi_step defined.")

18:26:40 | INFO    | grpo | evaluate_multi_step defined.


## 16 · Before / after multi-step evaluation

Runs the rich multi-step evaluator once with the **SFT-only** model (baseline) and once with the **final GRPO** model, then prints a delta table. Per-tier and overall comparisons are plotted by the matplotlib cell below.


In [29]:
def _flatten_metrics(m: RichMetrics, prefix: str) -> dict:
    out = {}
    for k, v in m.as_dict().items():
        if isinstance(v, dict):
            for tier, val in v.items():
                out[f"{prefix}/{k}/{tier}"] = val
        else:
            out[f"{prefix}/{k}"] = v
    return out


def _delta_metrics(before: RichMetrics, after: RichMetrics) -> dict:
    b, a = before.as_dict(), after.as_dict()
    delta: dict[str, float] = {}
    for k in a:
        if isinstance(a[k], dict):
            for tier, v in a[k].items():
                bv = b.get(k, {}).get(tier, 0.0)
                delta[f"eval/delta_{k}/{tier}"] = v - bv
        elif isinstance(a[k], (int, float)):
            delta[f"eval/delta_{k}"] = a[k] - b[k]
    return delta


EVAL_TASK_IDS = select_eval_task_ids(RESERVE_DS, DRIFT_TASK_IDS, cap=PIPE.eval_reserve_cap)
log.info("Multi-step eval on %d tasks (%d drift).",
         len(EVAL_TASK_IDS),
         sum(1 for t in EVAL_TASK_IDS if t in DRIFT_TASK_IDS))

# Release ENV_CLIENT's 8 WebSocket sessions so the server's MiniStack
# slots are free for GrpoPool. Without this, every eval episode dies
# instantly with ConnectionClosedOK / CAPACITY_REACHED because all 8
# slots are still held by the training reward client.
try:
    ENV_CLIENT.close()
    log.info("Closed ENV_CLIENT; waiting 3s for server to release 8 MiniStack slots.")
    await asyncio.sleep(3)
except Exception as e:
    log.warning("ENV_CLIENT.close() raised %s: %s", type(e).__name__, e)

# --- SFT baseline ---
log.info("Evaluating SFT baseline (multi-step)...")
sft_model, sft_tok = load_policy(MODEL, trainable=False)
sft_metrics = await evaluate_multi_step(
    ENV_BASE_URL, EVAL_TASK_IDS, TASK_MAP, DRIFT_TASK_IDS,
    sft_model, sft_tok, pool_size=PIPE.env_pool_size,
)
free_model(sft_model); del sft_tok
gc.collect(); torch.cuda.empty_cache()
(OUT_DIR / "baseline_multi_step.json").write_text(json.dumps(sft_metrics.as_dict(), indent=2))

# --- GRPO after ---
log.info("Evaluating GRPO-trained model (multi-step)...")
from unsloth import FastLanguageModel
if "final_trainer" not in globals() or "final_tok" not in globals():
    from peft import PeftModel
    _grpo_dir = OUT_DIR / "grpo_adapter"
    log.info("final_trainer not in globals; reloading GRPO adapter from %s...", _grpo_dir)
    _base, final_tok = FastLanguageModel.from_pretrained(
        model_name=MODEL.base_model,
        max_seq_length=MODEL.max_seq_length,
        load_in_4bit=True,
    )
    _grpo_model = PeftModel.from_pretrained(_base, str(_grpo_dir), is_trainable=False)
    class _TrainerShim:
        pass
    final_trainer = _TrainerShim()
    final_trainer.model = _grpo_model
FastLanguageModel.for_inference(final_trainer.model)
grpo_metrics = await evaluate_multi_step(
    ENV_BASE_URL, EVAL_TASK_IDS, TASK_MAP, DRIFT_TASK_IDS,
    final_trainer.model, final_tok, pool_size=PIPE.env_pool_size,
)
(OUT_DIR / "grpo_multi_step.json").write_text(json.dumps(grpo_metrics.as_dict(), indent=2))

deltas = _delta_metrics(sft_metrics, grpo_metrics)
log.info("SFT vs GRPO deltas: %s",
         {k: round(v, 4) for k, v in deltas.items()
          if isinstance(v, (int, float)) and "/" not in k})

def _render_row(name, b, a):
    return f"| {name:<26} | {b:>12.3f} | {a:>12.3f} | {a - b:+.3f} |"

print("\n| Metric                     | SFT baseline | GRPO        | Delta  |")
print("|----------------------------|-------------:|------------:|-------:|")
print(_render_row("overall_success_rate",  sft_metrics.overall_success_rate,  grpo_metrics.overall_success_rate))
print(_render_row("overall_reward_mean",   sft_metrics.overall_reward_mean,   grpo_metrics.overall_reward_mean))
print(_render_row("hints_per_solved",      sft_metrics.hints_per_solved,      grpo_metrics.hints_per_solved))
print(_render_row("recovery_rate",         sft_metrics.recovery_rate,         grpo_metrics.recovery_rate))
print(_render_row("drift_repair_rate",     sft_metrics.drift_repair_rate,     grpo_metrics.drift_repair_rate))
print(_render_row("steps_to_solve",        sft_metrics.steps_to_solve,        grpo_metrics.steps_to_solve))
print(_render_row("destructive_fail_rate", sft_metrics.destructive_fail_rate, grpo_metrics.destructive_fail_rate))

for tier in sorted(set(sft_metrics.success_by_tier) | set(grpo_metrics.success_by_tier)):
    b = sft_metrics.success_by_tier.get(tier, 0.0)
    a = grpo_metrics.success_by_tier.get(tier, 0.0)
    print(_render_row(f"success[{tier}]", b, a))

18:26:42 | INFO    | grpo | Multi-step eval on 109 tasks (9 drift).
18:26:43 | INFO    | grpo | Closed ENV_CLIENT; waiting 3s for server to release 8 MiniStack slots.
18:26:46 | INFO    | grpo | Evaluating SFT baseline (multi-step)...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
18:27:05 | INFO    | grpo | multi-step eval: 109 tasks in 14 chunks (pool_size=8)
18:27:05 | INFO    | server.services.curriculum | Loaded 25 warmup tasks total
18:27:05 | INFO    | server.services.curriculum | Curriculum initialised — starting at warmup with 25 tasks
18:27:06 | INFO    | scripts.grpo_pool | Grpo

## 17 · Qualitative rollouts — one task per tier

A small set of showable transcripts (task, generated command, env output, reward). Judges love seeing actual agent behaviour, not just numbers.

In [30]:
async def qualitative_rollouts(model, tokenizer, task_map: dict[int, Task],
                                drift_ids: set[int],
                                samples_per_tier: int = 1) -> list[dict]:
    """Pick one task per difficulty tier and print a full rollout transcript."""
    per_tier: dict[str, list[Task]] = defaultdict(list)
    for t in task_map.values():
        per_tier[t.difficulty.value].append(t)
    chosen: list[Task] = []
    for tier in ["warmup", "beginner", "intermediate", "advanced", "expert"]:
        if per_tier.get(tier):
            chosen.extend(per_tier[tier][:samples_per_tier])

    transcripts = []
    async with GrpoPool(base_url=ENV_BASE_URL, size=min(len(chosen), PIPE.env_pool_size)) as pool:
        for env, task in zip(pool.envs, chosen):
            ep = await run_episode(env, model, tokenizer, task, drift_ids, max_steps=8)
            transcripts.append({
                "tier": task.difficulty.value,
                "task_id": int(task.task_id),
                "description": task.description,
                "achieved": ep.achieved,
                "terminal_reward": ep.terminal_reward,
                "steps_taken": ep.steps_taken,
            })
    return transcripts


# Reload the GRPO adapter from disk if final_trainer was purged by an
# earlier VRAM reclaim (e.g. the Optuna cell). Same fallback as the
# multi-step eval cell: the final-run cell persists the adapter to
# OUT_DIR/"grpo_adapter", so that's the source of truth.
if "final_trainer" not in globals() or "final_tok" not in globals():
    from unsloth import FastLanguageModel
    from peft import PeftModel
    _grpo_dir = OUT_DIR / "grpo_adapter"
    print(f"final_trainer not in globals; reloading GRPO adapter from {_grpo_dir}...")
    _base, final_tok = FastLanguageModel.from_pretrained(
        model_name=MODEL.base_model,
        max_seq_length=MODEL.max_seq_length,
        load_in_4bit=True,
    )
    _grpo_model = PeftModel.from_pretrained(_base, str(_grpo_dir), is_trainable=False)
    class _TrainerShim:
        pass
    final_trainer = _TrainerShim()
    final_trainer.model = _grpo_model

qual = await qualitative_rollouts(final_trainer.model, final_tok, TASK_MAP, DRIFT_TASK_IDS)
print(json.dumps(qual, indent=2, default=str))
(OUT_DIR / "qualitative_rollouts.json").write_text(json.dumps(qual, indent=2, default=str))

18:59:10 | INFO    | server.services.curriculum | Loaded 25 warmup tasks total
18:59:10 | INFO    | server.services.curriculum | Curriculum initialised — starting at warmup with 25 tasks
18:59:11 | INFO    | scripts.grpo_pool | GrpoPool connected: 5 sessions against https://sizzing-aws-rl-env.hf.space
[
  {
    "tier": "warmup",
    "task_id": 0,
    "description": "List all S3 buckets in the environment.",
    "achieved": false,
    "terminal_reward": 0.0,
    "steps_taken": 8
  },
  {
    "tier": "beginner",
    "task_id": 6,
    "description": "Create an S3 bucket named 'my-test-bucket'.",
    "achieved": true,
    "terminal_reward": 1.0,
    "steps_taken": 1
  },
  {
    "tier": "intermediate",
    "task_id": 11,
    "description": "Create an S3 bucket named 'data-pipeline' and upload a file to it.",
    "achieved": true,
    "terminal_reward": 1.0,
    "steps_taken": 2
  },
  {
    "tier": "advanced",
    "task_id": 15,
    "description": "Create a Lambda function 'processor' with

1347

## 17.5 · Plot training curves and SFT vs GRPO comparison

Emits matplotlib PNGs into `OUT_DIR / "graphs/"` (also zipped as `graphs.zip` for
easy download):

1. `01_optuna_history.png` – per-trial objective + best-so-far.
2. `02_optuna_hparams.png` – hparam vs. objective scatter (one subplot per knob).
3. `03_optuna_trial_curves.png` – training loss / reward / reward_std / KL for
   every trial, with the winning trial highlighted.
4. `04_final_run_curves.png` – same four metrics for the final GRPO run on the
   best Optuna config.
5. `05_sft_vs_grpo_scalar.png` – scalar comparison (success, reward, recovery,
   drift repair, hints/solved, steps-to-solve, destructive-fail).
6. `06_success_by_tier.png` / `07_reward_by_tier.png` – per-difficulty-tier
   bars, SFT vs SFT+GRPO.

Each section is guarded so a missing artifact (e.g. Optuna was skipped, or
multi-step eval hasn\'t run yet) only skips that plot instead of failing.


In [31]:
import json
import math
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


GRAPHS_DIR = OUT_DIR / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

# Canonical tier ordering so bar plots read hardest-to-easiest consistently.
_TIER_ORDER = ["warmup", "beginner", "intermediate", "advanced", "expert"]


def _tier_key(t: str) -> int:
    return _TIER_ORDER.index(t) if t in _TIER_ORDER else 99


def _load_log_history(run_dir: Path) -> list[dict]:
    """Prefer a top-level trainer_state.json (written by trial_objective) and
    fall back to the latest checkpoint-N/trainer_state.json (the layout TRL
    uses when save_strategy != "no", i.e. the final run)."""
    top = run_dir / "trainer_state.json"
    if top.exists():
        return json.loads(top.read_text()).get("log_history", [])
    ckpts = sorted(
        (d for d in run_dir.glob("checkpoint-*") if d.is_dir()),
        key=lambda d: int(d.name.split("-")[-1]),
    )
    if not ckpts:
        return []
    state_path = ckpts[-1] / "trainer_state.json"
    if not state_path.exists():
        return []
    return json.loads(state_path.read_text()).get("log_history", [])


def _series(hist: list[dict], key: str) -> tuple[list[int], list[float]]:
    """Pull (step, value) pairs for a key from TRL's log_history, skipping
    entries that don't carry it (eval rows only have eval_* keys, etc.)."""
    xs, ys = [], []
    for row in hist:
        if key in row and isinstance(row[key], (int, float)):
            xs.append(row.get("step", len(xs)))
            ys.append(float(row[key]))
    return xs, ys


_saved: list[Path] = []


def _save(fig, name: str) -> None:
    out = GRAPHS_DIR / name
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    _saved.append(out)
    print(f"  saved {out.name}")


# ---------- 1) Optuna optimization history ----------
try:
    _trials = [t for t in study.trials if t.value is not None]
    if not _trials:
        raise RuntimeError("no completed Optuna trials")
    _nums = [t.number for t in _trials]
    _vals = [t.value  for t in _trials]
    _best = []
    _cur  = -math.inf
    for v in _vals:
        _cur = max(_cur, v)
        _best.append(_cur)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(_nums, _vals, "o-", label="trial env_reward_mean")
    ax.plot(_nums, _best, "r--", label="best so far")
    ax.scatter([study.best_trial.number], [study.best_value],
               s=120, facecolors="none", edgecolors="red", linewidths=2,
               label=f"best (trial {study.best_trial.number})")
    ax.set_xlabel("trial"); ax.set_ylabel("val env_reward_mean")
    ax.set_title("Optuna optimization history")
    ax.legend(); ax.grid(alpha=0.3)
    _save(fig, "01_optuna_history.png")
except Exception as e:
    print(f"  skipped 01_optuna_history.png: {e}")


# ---------- 2) Hparam vs objective scatter ----------
try:
    _trials = [t for t in study.trials if t.value is not None]
    if not _trials:
        raise RuntimeError("no completed Optuna trials")
    _params = ["learning_rate", "beta", "temperature"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, p in zip(axes.flat, _params):
        xs = [t.params[p] for t in _trials if p in t.params]
        ys = [t.value     for t in _trials if p in t.params]
        if not xs:
            ax.set_visible(False); continue
        ax.scatter(xs, ys, s=35)
        # Highlight best trial
        if p in study.best_params:
            ax.scatter([study.best_params[p]], [study.best_value],
                       s=120, facecolors="none", edgecolors="red",
                       linewidths=2, label="best")
            ax.legend(fontsize=8)
        if p == "learning_rate":
            ax.set_xscale("log")
        ax.set_xlabel(p); ax.set_ylabel("env_reward_mean")
        ax.grid(alpha=0.3)
    fig.suptitle("Hyperparameter vs objective (Optuna)")
    fig.tight_layout()
    _save(fig, "02_optuna_hparams.png")
except Exception as e:
    print(f"  skipped 02_optuna_hparams.png: {e}")


# ---------- 3) Per-trial training curves ----------
try:
    _trials = [t for t in study.trials if t.value is not None]
    if not _trials:
        raise RuntimeError("no completed Optuna trials")
    _metrics = [("loss", "training loss"),
                ("reward", "reward mean"),
                ("reward_std", "reward std (GRPO group)"),
                ("kl", "KL to reference")]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    _best_num = study.best_trial.number if study.best_trial else None
    _plotted_any = False
    for ax, (key, title) in zip(axes.flat, _metrics):
        for t in _trials:
            trial_dir = OUT_DIR / f"optuna/trial-{t.number}"
            hist = _load_log_history(trial_dir)
            xs, ys = _series(hist, key)
            if not xs:
                continue
            _plotted_any = True
            if t.number == _best_num:
                ax.plot(xs, ys, color="tab:red", lw=2.2,
                        label=f"trial {t.number} (best)")
            else:
                ax.plot(xs, ys, alpha=0.45, lw=1.2,
                        label=f"trial {t.number}")
        ax.set_title(title); ax.set_xlabel("step")
        ax.grid(alpha=0.3)
    if not _plotted_any:
        raise RuntimeError("no per-trial log_history on disk — re-run "
                           "Optuna after the trial_objective patch")
    # One legend for the whole figure
    handles, labels = axes.flat[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="lower center",
                   ncol=min(6, len(handles)), fontsize=8,
                   bbox_to_anchor=(0.5, -0.02))
    fig.suptitle("Optuna trial training curves")
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    _save(fig, "03_optuna_trial_curves.png")
except Exception as e:
    print(f"  skipped 03_optuna_trial_curves.png: {e}")


# ---------- 4) Final GRPO run curves ----------
try:
    hist = _load_log_history(FINAL_RUN_DIR)
    if not hist:
        raise RuntimeError(f"no log_history under {FINAL_RUN_DIR}")
    _metrics = [("loss", "training loss"),
                ("reward", "reward mean"),
                ("reward_std", "reward std (GRPO group)"),
                ("kl", "KL to reference"),
                ("learning_rate", "learning rate"),
                ("completion_length", "completion length (tokens)")]
    fig, axes = plt.subplots(3, 2, figsize=(13, 10))
    for ax, (key, title) in zip(axes.flat, _metrics):
        xs, ys = _series(hist, key)
        if not xs:
            ax.set_visible(False); continue
        ax.plot(xs, ys, color="tab:blue")
        ax.set_title(title); ax.set_xlabel("step")
        ax.grid(alpha=0.3)
        if key == "learning_rate":
            ax.set_yscale("log")
    fig.suptitle("Final GRPO run — best Optuna config")
    fig.tight_layout()
    _save(fig, "04_final_run_curves.png")
except Exception as e:
    print(f"  skipped 04_final_run_curves.png: {e}")


# ---------- 5) SFT vs GRPO scalar comparison (multi-step eval) ----------
try:
    sft  = json.loads((OUT_DIR / "baseline_multi_step.json").read_text())
    grpo = json.loads((OUT_DIR / "grpo_multi_step.json").read_text())
    _keys = ["overall_success_rate", "overall_reward_mean",
             "recovery_rate", "drift_repair_rate",
             "hints_per_solved", "steps_to_solve",
             "destructive_fail_rate"]
    x = np.arange(len(_keys))
    fig, ax = plt.subplots(figsize=(12, 5))
    w = 0.38
    sft_vals  = [float(sft.get(k, 0.0))  for k in _keys]
    grpo_vals = [float(grpo.get(k, 0.0)) for k in _keys]
    b1 = ax.bar(x - w/2, sft_vals,  width=w, label="SFT only",   color="tab:gray")
    b2 = ax.bar(x + w/2, grpo_vals, width=w, label="SFT + GRPO", color="tab:blue")
    for bars, vals in ((b1, sft_vals), (b2, grpo_vals)):
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, v,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(_keys, rotation=25, ha="right")
    ax.set_title("Multi-step eval: SFT vs SFT+GRPO")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    _save(fig, "05_sft_vs_grpo_scalar.png")
except Exception as e:
    print(f"  skipped 05_sft_vs_grpo_scalar.png: {e}")


# ---------- 6/7) Per-tier SFT vs GRPO ----------
def _per_tier_plot(metric_key: str, ylabel: str, fname: str) -> None:
    try:
        sft  = json.loads((OUT_DIR / "baseline_multi_step.json").read_text())
        grpo = json.loads((OUT_DIR / "grpo_multi_step.json").read_text())
        sft_t  = sft.get(metric_key, {}) or {}
        grpo_t = grpo.get(metric_key, {}) or {}
        tiers = sorted(set(sft_t) | set(grpo_t), key=_tier_key)
        if not tiers:
            raise RuntimeError(f"no tiers in {metric_key}")
        x = np.arange(len(tiers))
        fig, ax = plt.subplots(figsize=(9, 4.5))
        w = 0.38
        ax.bar(x - w/2, [float(sft_t.get(t, 0.0))  for t in tiers],
               width=w, label="SFT only",   color="tab:gray")
        ax.bar(x + w/2, [float(grpo_t.get(t, 0.0)) for t in tiers],
               width=w, label="SFT + GRPO", color="tab:blue")
        ax.set_xticks(x); ax.set_xticklabels(tiers)
        ax.set_title(f"{metric_key} by tier")
        ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()
        _save(fig, fname)
    except Exception as e:
        print(f"  skipped {fname}: {e}")


_per_tier_plot("success_by_tier", "success rate",       "06_success_by_tier.png")
_per_tier_plot("reward_by_tier",  "mean terminal reward", "07_reward_by_tier.png")


# ---------- Zip for easy download ----------
if _saved:
    _zip_base = str(OUT_DIR / "graphs")
    _zip_path = shutil.make_archive(_zip_base, "zip", root_dir=GRAPHS_DIR)
    print(f"\nGraphs: {GRAPHS_DIR} ({len(_saved)} PNGs)")
    print(f"Zipped: {_zip_path}")
else:
    print("\nNo graphs were produced — every section was skipped.")


  saved 01_optuna_history.png
  saved 02_optuna_hparams.png
  saved 03_optuna_trial_curves.png
  saved 04_final_run_curves.png
  saved 05_sft_vs_grpo_scalar.png
  saved 06_success_by_tier.png
  saved 07_reward_by_tier.png

Graphs: /content/out/graphs (7 PNGs)
Zipped: /content/out/graphs.zip


## 18 · Publish the GRPO adapter to a new HF Hub repo

Pushes **only** to `Sizzing/aws-rl-grpo-qwen25coder3b-adapter`. The existing SFT adapter repo `Sizzing/aws-rl-sft-qwen25coder3b-adapter` is never touched — both coexist on Hub so reviewers can load either and compare side by side.

The model card notes the lineage (`base_model = SFT adapter`) so anyone opening the repo on Hub sees immediately that this is a second-stage RL fine-tune.

In [32]:
from datetime import datetime, timezone


def write_model_card(adapter_dir: Path, model_spec: ModelSpec,
                     cfg: TrainingConfig, grpo: RichMetrics, sft: RichMetrics) -> Path:
    """Emit a README.md for the pushed repo documenting training recipe + metrics."""
    card = f"""---
library_name: peft
base_model: {model_spec.sft_adapter}
pipeline_tag: text-generation
tags: [grpo, lora, aws, reinforcement-learning]
---

# aws-rl-grpo-qwen25coder3b-adapter

GRPO (Group Relative Policy Optimization) LoRA adapter continuing from
[`{model_spec.sft_adapter}`](https://huggingface.co/{model_spec.sft_adapter}).
Trained single-step with live AWS RL env rewards; evaluated multi-step.

## How to load

```python
from unsloth import FastLanguageModel
from peft import PeftModel

base, tok = FastLanguageModel.from_pretrained(
    "{model_spec.base_model}", max_seq_length={model_spec.max_seq_length}, load_in_4bit=True,
)
model = PeftModel.from_pretrained(base, "{model_spec.grpo_adapter}")
FastLanguageModel.for_inference(model)
```

## Training recipe

| Knob                  | Value              |
|-----------------------|--------------------|
| learning_rate         | {cfg.learning_rate:.2e}   |
| beta (KL coef)        | {cfg.beta:.3f}             |
| num_generations (G)   | {cfg.num_generations}                  |
| temperature           | {cfg.temperature:.2f}              |
| max_completion_length | {cfg.max_completion_length}                |
| per-device batch      | {cfg.per_device_train_batch_size} x {cfg.gradient_accumulation_steps} accum       |

## Multi-step eval (reserve + drift)

| Metric                | SFT baseline | GRPO   | Delta   |
|-----------------------|-------------:|-------:|--------:|
| overall_success_rate  | {sft.overall_success_rate:.3f}        | {grpo.overall_success_rate:.3f}  | {grpo.overall_success_rate - sft.overall_success_rate:+.3f} |
| overall_reward_mean   | {sft.overall_reward_mean:.3f}        | {grpo.overall_reward_mean:.3f}  | {grpo.overall_reward_mean - sft.overall_reward_mean:+.3f} |
| hints_per_solved      | {sft.hints_per_solved:.3f}        | {grpo.hints_per_solved:.3f}  | {grpo.hints_per_solved - sft.hints_per_solved:+.3f} |
| recovery_rate         | {sft.recovery_rate:.3f}        | {grpo.recovery_rate:.3f}  | {grpo.recovery_rate - sft.recovery_rate:+.3f} |
| drift_repair_rate     | {sft.drift_repair_rate:.3f}        | {grpo.drift_repair_rate:.3f}  | {grpo.drift_repair_rate - sft.drift_repair_rate:+.3f} |
| steps_to_solve        | {sft.steps_to_solve:.3f}        | {grpo.steps_to_solve:.3f}  | {grpo.steps_to_solve - sft.steps_to_solve:+.3f} |

Trained {datetime.now(timezone.utc).isoformat(timespec='minutes')} on {RT.gpu_name}.
"""
    card_path = adapter_dir / "README.md"
    card_path.write_text(card)
    return card_path


# Write card locally, then push both adapter files + card
adapter_local = OUT_DIR / "grpo_adapter"
write_model_card(adapter_local, MODEL, best_cfg, grpo_metrics, sft_metrics)

commit_msg = f"GRPO run {datetime.now(timezone.utc).isoformat(timespec='minutes')}"
final_trainer.model.push_to_hub(
    MODEL.grpo_adapter, private=True, token=HF_TOKEN, commit_message=commit_msg,
)
final_tok.push_to_hub(
    MODEL.grpo_adapter, private=True, token=HF_TOKEN, commit_message=commit_msg,
)
HfApi(token=HF_TOKEN).upload_file(
    path_or_fileobj=str(adapter_local / "README.md"),
    path_in_repo="README.md",
    repo_id=MODEL.grpo_adapter,
    commit_message=commit_msg,
)
print(f"Pushed to https://huggingface.co/{MODEL.grpo_adapter}")
print(f"SFT adapter at {MODEL.sft_adapter} untouched — both models available on Hub.")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|1         |  563kB / 29.5MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp7c5v_5a_/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Pushed to https://huggingface.co/Sizzing/aws-rl-grpo-qwen25coder3b-adapter
SFT adapter at Sizzing/aws-rl-sft-qwen25coder3b-adapter untouched — both models available on Hub.


## 19 · Clean shutdown + artifact bundle

Releases the GPU, tars `OUT_DIR` into a single downloadable archive (Colab `files.download`). Nothing else needs to be killed — the env server is hosted externally.


In [33]:
import shutil
from pathlib import Path

# Verify graphs were exported
graphs_dir = OUT_DIR / "graphs"
if graphs_dir.exists():
    pngs = sorted(graphs_dir.glob("*.png"))
    log.info("Found %d PNGs in %s:", len(pngs), graphs_dir)
    for p in pngs:
        log.info("  %s (%d KB)", p.name, p.stat().st_size // 1024)
else:
    log.warning("No graphs dir at %s — run the matplotlib plot cell first.", graphs_dir)

# Release GPU
try:
    free_model(final_trainer)
    del final_trainer, final_model, final_tok
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()

# Build a single zip of OUT_DIR (grpo_adapter, JSON metrics, graphs/, etc.)
zip_path = shutil.make_archive(
    base_name=str(OUT_DIR.parent / OUT_DIR.name),
    format="zip",
    root_dir=OUT_DIR,
)
log.info("Zip bundle: %s (%.1f MB)",
         zip_path, Path(zip_path).stat().st_size / 1e6)
log.info("HF Hub: SFT=https://huggingface.co/%s GRPO=https://huggingface.co/%s",
         MODEL.sft_adapter, MODEL.grpo_adapter)

if IS_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        log.warning("Colab auto-download skipped: %s. Download manually from %s",
                    e, zip_path)

19:05:30 | INFO    | grpo | Found 10 PNGs in /content/out/graphs:
19:05:30 | INFO    | grpo |   00_optuna_history.png (38 KB)
19:05:30 | INFO    | grpo |   00_optuna_importances.png (39 KB)
19:05:30 | INFO    | grpo |   00_optuna_parallel.png (97 KB)
19:05:30 | INFO    | grpo |   01_optuna_history.png (53 KB)
19:05:30 | INFO    | grpo |   02_optuna_hparams.png (52 KB)
19:05:30 | INFO    | grpo |   03_optuna_trial_curves.png (270 KB)
19:05:30 | INFO    | grpo |   04_final_run_curves.png (254 KB)
19:05:30 | INFO    | grpo |   05_sft_vs_grpo_scalar.png (84 KB)
19:05:30 | INFO    | grpo |   06_success_by_tier.png (33 KB)
19:05:30 | INFO    | grpo |   07_reward_by_tier.png (34 KB)
19:05:41 | INFO    | grpo | Zip bundle: /content/out.zip (122.2 MB)
19:05:41 | INFO    | grpo | HF Hub: SFT=https://huggingface.co/Sizzing/aws-rl-sft-qwen25coder3b-adapter GRPO=https://huggingface.co/Sizzing/aws-rl-grpo-qwen25coder3b-adapter


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>